In [ ]:
#Cleaning files
import pandas as pd
from pathlib import Path

# ─────────── USER PARAMETERS ───────────
input_dir = Path('/content/drive/MyDrive/ColabNotebooks/cvs/cvs')
output_dir = Path('/content/drive/MyDrive/ColabNotebooks/cleaned2')
output_dir.mkdir(parents=True, exist_ok=True)
# ────────────────────────────────────────

def clean_file(in_path: Path, out_path: Path):
    df = pd.read_csv(in_path)

    # 1. Impute missing values by linear interpolation (then fill any ends)
    df = df.interpolate(limit_direction='both').fillna(method='bfill').fillna(method='ffill')

    # 2. Remove outliers via IQR, but only if that column has enough variability
    num_cols = df.select_dtypes(include='number').columns
    for col in num_cols:
        series = df[col]
        Q1, Q3 = series.quantile([0.25, 0.75])
        IQR = Q3 - Q1

        # If removing outliers would eliminate >90% of data, skip filtering
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        mask = series.between(lower, upper)
        if mask.sum() / len(series) >= 0.1:
            df = df[mask]

    # 3. If entire frame got dropped (unlikely now), just save the interpolated version
    if df.empty:
        df = pd.read_csv(in_path).interpolate(limit_direction='both').fillna(method='bfill').fillna(method='ffill')

    # 4. Save cleaned CSV
    df.to_csv(out_path, index=False)

# Run cleaning
for csv_file in sorted(input_dir.glob("*.csv")):
    clean_file(csv_file, output_dir / csv_file.name)

# Report
cleaned_files = [p.name for p in output_dir.glob("*.csv")]
print("Cleaned files saved to:", output_dir)
print(cleaned_files)


In [ ]:
#trimmed files or cleaned files
import pandas as pd
import numpy as np
from scipy.fft import rfft, rfftfreq
from pathlib import Path

input_dir = Path('/content/drive/MyDrive/ColabNotebooks/15p-trimmed_csvs')
output_dir = Path('/content/drive/MyDrive/ColabNotebooks/frequencydomain_')
output_dir.mkdir(parents=True, exist_ok=True)

sensor_cols = ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
meta_cols = ['case', 'run', 'VB', 'DOC', 'feed', 'material']

for csv_file in input_dir.glob("*.csv"):
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"Error reading {csv_file}: {e}")
        continue

    # Ensure at least 2 points by padding if necessary
    if len(df) < 2:
        df = pd.concat([df, df], ignore_index=True)[:2]  # Duplicate data to reach 2 points

    # Calculate sampling rate
    fs = 1.0
    if 'time' in df.columns and not df['time'].diff().dropna().empty:
        mean_dt = df['time'].diff().mean()
        if mean_dt > 0:
            fs = 1.0 / mean_dt

    N = len(df)
    freqs = rfftfreq(N, d=1/fs)
    out_df = pd.DataFrame({'frequency': freqs})

    for col in sensor_cols:
        data = df[col].values if col in df.columns else np.zeros(N)
        X = rfft(data)
        out_df[f'{col}_mag'] = np.abs(X)

    for col in meta_cols:
        out_df[col] = df[col].iloc[0] if col in df.columns else np.nan

    ordered_cols = meta_cols + ['frequency'] + [f'{col}_mag' for col in sensor_cols]
    out_df = out_df[ordered_cols]

    out_path = output_dir / f"freq_{csv_file.name}"
    out_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path.name}")

print("All files processed.")

In [ ]:
#for personal visualization
import pandas as pd
import matplotlib.pyplot as plt

# ─────────── USER PARAMETERS ───────────

raw_file = '/content/mill_000.csv'
cleaned_file = '/content/mill_000final.csv'
column_to_plot = 'AE_spindle'
# ───────────────────────────────────────

# Load data
df_raw = pd.read_csv(raw_file)
df_cleaned = pd.read_csv(cleaned_file)

# Check if column exists
if column_to_plot not in df_raw.columns or column_to_plot not in df_cleaned.columns:
    raise ValueError(f"Column '{column_to_plot}' not found in one or both files.")

# Plot
plt.figure(figsize=(12, 5))
plt.scatter(range(len(df_raw[column_to_plot])), df_raw[column_to_plot],
            label='Before Cleaning', alpha=0.5, s=10, color='red')
plt.scatter(range(len(df_cleaned[column_to_plot])), df_cleaned[column_to_plot],
            label='After Cleaning', alpha=0.6, s=10, color='green')

plt.title(f"Scatter Plot of '{column_to_plot}' Before and After Cleaning")
plt.xlabel("Sample Index")
plt.ylabel("Sensor Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#Calculating time domain features
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import skew, kurtosis, entropy

# ─────────── USER PARAMETERS ───────────
# Folder with cleaned CSV files
input_dir = Path('/content/drive/MyDrive/ColabNotebooks/15p-trimmed_csvs')

# Output file location
output_file = Path('/content/drive/MyDrive/ColabNotebooks/time_domain_features.csv')

# Sensor columns to extract features from
sensor_cols = ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']

# Metadata columns to keep
meta_cols = ['case', 'run', 'VB', 'time', 'DOC', 'feed', 'material']
# ───────────────────────────────────────

def zero_crossings(x):
    """Count zero crossings in signal."""
    return ((x[:-1] * x[1:]) < 0).sum()

def compute_features(x):
    """Compute 14 time-domain features from 1D array."""
    x = np.array(x)
    mean_val = np.mean(x)
    std_val = np.std(x)
    rms = np.sqrt(np.mean(np.square(x)))
    peak = np.max(np.abs(x))
    p2p = np.ptp(x)
    skewness = skew(x)
    kurt = kurtosis(x)
    crest = peak / rms if rms != 0 else np.nan
    impulse = peak / np.mean(np.abs(x)) if np.mean(np.abs(x)) != 0 else np.nan
    shape = rms / np.mean(np.abs(x)) if np.mean(np.abs(x)) != 0 else np.nan
    hist = np.histogram(x, bins=50, density=True)[0]
    entropy_val = entropy(hist + 1e-12)
    zcr = zero_crossings(x)
    energy = np.sum(np.square(x))
    var = np.var(x)

    return [mean_val, std_val, rms, peak, p2p, skewness, kurt, crest, impulse, shape, entropy_val, zcr, energy, var]

# Feature names for columns
feature_names = ['mean', 'std', 'rms', 'peak', 'p2p', 'skewness', 'kurtosis',
                 'crest', 'impulse', 'shape', 'entropy', 'zcr', 'energy', 'variance']

# Collect feature rows
rows = []

# Process each file
for file in sorted(input_dir.glob("*.csv")):
    df = pd.read_csv(file)
    if df.empty:
        continue

    # Start row with metadata
    row = df[meta_cols].iloc[0].to_dict()

    # Compute features for each sensor column
    for col in sensor_cols:
        if col in df.columns:
            feats = compute_features(df[col])
            for name, val in zip(feature_names, feats):
                row[f"{col}_{name}"] = val
        else:
            for name in feature_names:
                row[f"{col}_{name}"] = np.nan

    rows.append(row)

# Create and save final DataFrame
final_df = pd.DataFrame(rows)
final_df.to_csv(output_file, index=False)

print(f"✅ Feature file saved to: {output_file}")


In [ ]:
#Calculating frequency domain features
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import skew, kurtosis, entropy
from scipy.integrate import simpson

# ─────────── USER PARAMETERS ───────────
input_dir = Path('/content/drive/MyDrive/ColabNotebooks/frequencydomain_')
output_file = Path('/content/drive/MyDrive/ColabNotebooks/frequency_features.csv')

sensor_cols = ['smcAC_mag', 'smcDC_mag', 'vib_table_mag', 'vib_spindle_mag', 'AE_table_mag', 'AE_spindle_mag']
meta_cols = ['case', 'run', 'VB', 'DOC', 'feed', 'material']
# ───────────────────────────────────────

def zero_crossings(x):
    return ((x[:-1] * x[1:]) < 0).sum()

def extract_freq_features(freq, spectrum_raw):
    spectrum = np.abs(spectrum_raw)
    spectrum_sum = np.sum(spectrum)
    if spectrum_sum == 0 or np.isnan(spectrum_sum):
        return [np.nan] * 13

    spectrum /= spectrum_sum  # Normalize
    psd = spectrum ** 2

    band_power = simpson(psd, freq)
    peak_freq = freq[np.argmax(psd)]
    centroid = np.sum(freq * spectrum)
    rolloff_idx = np.argmax(np.cumsum(spectrum) >= 0.85)
    rolloff = freq[rolloff_idx] if rolloff_idx < len(freq) else np.nan
    bandwidth = np.sqrt(np.sum(((freq - centroid) ** 2) * spectrum))
    flatness = np.exp(np.mean(np.log(spectrum + 1e-12))) / (np.mean(spectrum) + 1e-12)
    spec_entropy = entropy(spectrum + 1e-12)
    spec_skew = skew(spectrum)
    spec_kurt = kurtosis(spectrum)
    rms_freq = np.sqrt(np.mean(spectrum ** 2))
    zcr = zero_crossings(spectrum_raw)
    dominant_freq = peak_freq

    return [
        band_power, peak_freq, centroid, rolloff, bandwidth,
        flatness, spec_entropy, spec_skew, spec_kurt,
        rms_freq, zcr, dominant_freq
    ]

feature_names = [
    'band_power', 'peak_freq', 'centroid', 'rolloff', 'bandwidth',
    'flatness', 'entropy', 'skewness', 'kurtosis',
    'rms', 'zcr_freq', 'dominant_freq'
]

rows = []

for file in sorted(input_dir.glob("*.csv")):
    df = pd.read_csv(file)
    if df.empty or 'frequency' not in df.columns:
        continue

    freq = df['frequency'].values

    # Group by metadata (assuming each file is one case)
    row = {}
    for meta in meta_cols:
        row[meta] = df[meta].iloc[0] if meta in df.columns else np.nan

    for col in sensor_cols:
        if col in df.columns:
            spectrum = df[col].values
            feats = extract_freq_features(freq, spectrum)
            for name, val in zip(feature_names, feats):
                row[f"{col}_{name}"] = val
        else:
            for name in feature_names:
                row[f"{col}_{name}"] = np.nan

    rows.append(row)

# Save results
final_df = pd.DataFrame(rows)
final_df.to_csv(output_file, index=False)
print(f"✅ Frequency features saved to: {output_file}")


In [ ]:
#getting top ranked features
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy import stats

# STEP 1: Load and Prepare Data
def load_and_prepare_data(time_path, freq_path):
    # Load datasets
    time_df = pd.read_csv(time_path)
    freq_df = pd.read_csv(freq_path)

    # Define meta + target columns
    meta_cols = ['case', 'run', 'DOC', 'time', 'feed', 'material']
    target_col = 'VB'

    # Merge datasets
    combined_df = pd.merge(
        time_df,
        freq_df.drop(columns=meta_cols + [target_col]),
        left_index=True,
        right_index=True
    )

    return combined_df, meta_cols, target_col

# STEP 2: Feature Selection with Stability Analysis
def select_stable_features(X, y, n_iterations=10):
    feature_importance_scores = []

    for i in range(n_iterations):
        # Random subsample
        X_sample, _, y_sample, _ = train_test_split(X, y, test_size=0.3, random_state=i)

        # Train RF on subsample
        rf = RandomForestRegressor(n_estimators=100, random_state=i)
        rf.fit(X_sample, y_sample)

        # Store importance scores
        importance = pd.DataFrame({
            'Feature': X.columns,
            'Importance': rf.feature_importances_
        })
        feature_importance_scores.append(importance)

    # Calculate mean and std of importance scores
    mean_importance = pd.concat(feature_importance_scores).groupby('Feature').mean()
    std_importance = pd.concat(feature_importance_scores).groupby('Feature').std()

    stability_scores = pd.DataFrame({
        'Feature': mean_importance.index,
        'Mean_Importance': mean_importance['Importance'],
        'Std_Importance': std_importance['Importance']
    })

    return stability_scores.sort_values('Mean_Importance', ascending=False)

# STEP 3: Model Training and Evaluation
def train_and_evaluate_model(X, y, feature_list=None):
    if feature_list is not None:
        X = X[feature_list]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

    # Grid search
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2']
    }

    grid_search = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )

    grid_search.fit(X_train_imp, y_train)
    best_rf = grid_search.best_estimator_

    # Evaluate
    y_pred = best_rf.predict(X_test_imp)
    scores = {
        'R2': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred)
    }

    return best_rf, scores, X_train_imp, X_test_imp, y_train, y_test

# STEP 4: Visualization Functions
def plot_feature_importance(importance_df, title, output_path=None):
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Relative_Importance', y='Feature', data=importance_df.head(44))
    plt.title(title)
    plt.xlabel('Relative Importance (%)')
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path)
    plt.show()

def plot_stability_analysis(stability_df, output_path=None):
    plt.figure(figsize=(12, 6))
    plt.errorbar(range(len(stability_df)),
                stability_df['Mean_Importance'],
                yerr=stability_df['Std_Importance'],
                fmt='o')
    plt.title('Feature Importance Stability Analysis')
    plt.xlabel('Feature Rank')
    plt.ylabel('Mean Importance (with std error)')
    if output_path:
        plt.savefig(output_path)
    plt.show()

# Main execution
def main():
    # Load data
    combined_df, meta_cols, target_col = load_and_prepare_data(
        '/content/drive/MyDrive/ColabNotebooks/time_domain_features.csv',
        '/content/drive/MyDrive/ColabNotebooks/frequency_features.csv'
    )

    # Prepare features and target
    X = combined_df.drop(columns=meta_cols + [target_col])
    y = combined_df[target_col]

    # Perform stability analysis
    stability_scores = select_stable_features(X, y)

    # Select top 44 stable features
    top_44_features = stability_scores.head(44)['Feature'].tolist()

    # Train model with selected features
    best_rf, scores, X_train, X_test, y_train, y_test = train_and_evaluate_model(X, y, top_44_features)

    # Calculate final feature importance
    final_importance = pd.DataFrame({
        'Feature': top_44_features,
        'Importance': best_rf.feature_importances_
    })
    final_importance['Relative_Importance'] = (final_importance['Importance'] -
                                             final_importance['Importance'].min()) / \
                                            (final_importance['Importance'].max() -
                                             final_importance['Importance'].min()) * 100

    # Save results
    final_importance.to_csv('top_44_features.csv', index=False)

    # Create combined dataset with selected features
    selected_features_df = combined_df[meta_cols + [target_col] + top_44_features]
    selected_features_df.to_csv('Top44_Features_with_Metadata.csv', index=False)

    # Print results
    print("\n=== Model Performance ===")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")

    print("\n=== Top 10 Features ===")
    print(final_importance.head(10))

    # Create visualizations
    plot_feature_importance(final_importance, 'Top 44 Features Importance', 'feature_importance.png')
    plot_stability_analysis(stability_scores.head(44), 'feature_stability.png')

if __name__ == "__main__":
    main()

In [ ]:
# First, let's train a Random Forest on ALL features to get the base importance
def get_full_feature_importance(X, y):
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    return pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)

# Load the data
combined_df, meta_cols, target_col = load_and_prepare_data(
    '/content/drive/MyDrive/ColabNotebooks/time_domain_features.csv',
    '/content/drive/MyDrive/ColabNotebooks/frequency_features.csv'
)

# Get all features
X_full = combined_df.drop(columns=meta_cols + [target_col])
y = combined_df[target_col]

# Get importance for ALL features
all_features_importance = get_full_feature_importance(X_full, y)

# Load the top 44 features
top_44_features = pd.read_csv('/content/top_44_features.csv')

# Calculate the true percentage
total_importance = all_features_importance['Importance'].sum()
top_44_sum = all_features_importance[all_features_importance['Feature'].isin(top_44_features['Feature'])]['Importance'].sum()
percent_top_44 = (top_44_sum / total_importance) * 100

print("\n📌 Feature Importance Analysis:")
print(f"Total number of features: {len(all_features_importance)}")
print(f"Number of selected features: {len(top_44_features)}")
print(f"\n🔢 Sum of Importance (Top 44): {top_44_sum:.4f}")
print(f"🔢 Total Importance (All Features): {total_importance:.4f}")
print(f"📊 Top 44 contribute: {percent_top_44:.2f}% of total importance")

# Optional: Show distribution of importance
plt.figure(figsize=(10, 6))
plt.plot(range(len(all_features_importance)), all_features_importance['Importance'].cumsum() * 100)
plt.axvline(x=44, color='r', linestyle='--', label='Top 44 cutoff')
plt.xlabel('Number of Features')
plt.ylabel('Cumulative Importance (%)')
plt.title('Cumulative Feature Importance')
plt.legend()
plt.grid(True)
plt.show()

# Print importance distribution
print("\nImportance distribution by feature groups:")
print(f"Top 8 features: {(all_features_importance.head(8)['Importance'].sum() / total_importance * 100):.2f}%")
print(f"Top 20 features: {(all_features_importance.head(20)['Importance'].sum() / total_importance * 100):.2f}%")
print(f"Top 30 features: {(all_features_importance.head(30)['Importance'].sum() / total_importance * 100):.2f}%")
print(f"Top 44 features: {(all_features_importance.head(44)['Importance'].sum() / total_importance * 100):.2f}%")
print(f"Remaining {len(all_features_importance) - 44} features: {(all_features_importance.tail(len(all_features_importance) - 44)['Importance'].sum() / total_importance * 100):.2f}%")

In [ ]:
#Determining threshold for number of features using elbow method
!pip install -q kneed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from kneed import KneeLocator
import seaborn as sns

def find_optimal_threshold(stability_scores, n_clusters=3, sensitivity=1.0):
    """Find optimal threshold using combined elbow, clustering, and curvature methods"""
    # Get sorted importance values
    importances = stability_scores['Mean_Importance'].sort_values(ascending=False).values

    # Convert to 2D array for clustering
    X = importances.reshape(-1, 1)

    # Method 1: K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(X)
    cluster_centers = np.sort(kmeans.cluster_centers_.flatten())[::-1]
    kmeans_threshold = cluster_centers[1]  # Between top two clusters

    # Method 2: Modified elbow detection
    x = np.arange(len(importances))
    knee = KneeLocator(x, importances,
                      curve='convex',
                      direction='decreasing',
                      S=sensitivity)
    elbow_threshold = importances[knee.knee] if knee.knee else None

    # Method 3: Cumulative importance cutoff
    cumulative = np.cumsum(importances)
    cum_threshold = importances[np.argmax(cumulative >= 0.95 * cumulative[-1])]

    # Combine results
    thresholds = [t for t in [kmeans_threshold, elbow_threshold, cum_threshold] if t is not None]
    return np.mean(thresholds)

# Load the stability scores
combined_df, meta_cols, target_col = load_and_prepare_data(
    '/content/drive/MyDrive/ColabNotebooks/time_domain_features.csv',
    '/content/drive/MyDrive/ColabNotebooks/frequency_features.csv'
)
X = combined_df.drop(columns=meta_cols + [target_col])
stability_scores = select_stable_features(X, combined_df[target_col])

# Find optimal threshold
optimal_threshold = find_optimal_threshold(stability_scores, n_clusters=3, sensitivity=1.5)

# Select features based on threshold
selected_features = stability_scores[stability_scores['Mean_Importance'] >= optimal_threshold]

# Create visualization
plt.figure(figsize=(12, 6))

# Plot importance curve
importances = stability_scores['Mean_Importance'].values
x = np.arange(len(importances))
plt.plot(x, importances, 'b-', label='Feature Importance')

# Plot error bars
plt.errorbar(x, importances,
            yerr=stability_scores['Std_Importance'].values,
            fmt='none', color='lightblue', alpha=0.3)

# Plot thresholds
plt.axhline(y=optimal_threshold, color='r', linestyle='--',
           label=f'Optimal Threshold ({optimal_threshold:.4f})')
plt.axvline(x=len(selected_features), color='g', linestyle=':',
           label=f'Selected Features ({len(selected_features)})')

# Add annotations
plt.text(len(importances)*0.7, optimal_threshold*1.1,
        f'Threshold: {optimal_threshold:.4f}',
        color='red', fontsize=12)

plt.text(len(selected_features), importances[0]*0.8,
        f'Selected: {len(selected_features)} features',
        rotation=90, color='green', fontsize=12)

# Formatting
plt.title('Optimized Feature Importance Threshold Detection\n(With Stability Analysis)')
plt.xlabel('Feature Rank')
plt.ylabel('Mean Importance Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Print results
print("\n=== Threshold Analysis ===")
print(f"🔍 Optimal threshold: {optimal_threshold:.4f}")
print(f"✅ Number of selected features: {len(selected_features)}")

# Calculate importance coverage
total_importance = stability_scores['Mean_Importance'].sum()
selected_importance = selected_features['Mean_Importance'].sum()
importance_coverage = (selected_importance / total_importance) * 100

print(f"📊 Selected features cover {importance_coverage:.2f}% of total importance")

print("\n📋 Selected features:")
print(selected_features[['Feature', 'Mean_Importance', 'Std_Importance']].to_string(index=False))

# Compare with fixed top 44
top_44_importance = stability_scores.head(44)['Mean_Importance'].sum()
top_44_coverage = (top_44_importance / total_importance) * 100

print(f"\n📊 Top 44 features cover {top_44_coverage:.2f}% of total importance")
print(f"Difference in coverage: {abs(importance_coverage - top_44_coverage):.2f}%")

# Save results
selected_features.to_csv('optimal_threshold_features.csv', index=False)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from kneed import KneeLocator

def find_relaxed_threshold_for_44(stability_scores, target_n_features=44):
    """Find threshold that gives closest to 44 features"""
    importances = stability_scores['Mean_Importance'].sort_values(ascending=False).values

    # Binary search to find threshold that gives closest to 44 features
    left = min(importances)
    right = max(importances)
    best_threshold = None
    best_diff = float('inf')

    for _ in range(100):
        mid = (left + right) / 2
        n_features = sum(importances >= mid)

        if abs(n_features - target_n_features) < best_diff:
            best_threshold = mid
            best_diff = abs(n_features - target_n_features)

        if n_features > target_n_features:
            left = mid
        else:
            right = mid

    return best_threshold

# Load and prepare data
combined_df, meta_cols, target_col = load_and_prepare_data(
    '/content/drive/MyDrive/ColabNotebooks/time_domain_features.csv',
    '/content/drive/MyDrive/ColabNotebooks/frequency_features.csv'
)
X = combined_df.drop(columns=meta_cols + [target_col])
stability_scores = select_stable_features(X, combined_df[target_col])

# Add relative importance
max_importance = stability_scores['Mean_Importance'].max()
min_importance = stability_scores['Mean_Importance'].min()
stability_scores['Relative_Importance'] = ((stability_scores['Mean_Importance'] - min_importance) /
                                         (max_importance - min_importance) * 100)

# Calculate thresholds
original_threshold = find_optimal_threshold(stability_scores)
relaxed_threshold = find_relaxed_threshold_for_44(stability_scores)

# Select features
original_selected = stability_scores[stability_scores['Mean_Importance'] >= original_threshold]
relaxed_selected = stability_scores[stability_scores['Mean_Importance'] >= relaxed_threshold]

# Visualization
plt.figure(figsize=(14, 7))

# Plot importance curve with error bars
importances = stability_scores['Mean_Importance'].values
std_importances = stability_scores['Std_Importance'].values
x = np.arange(len(importances))

plt.errorbar(x, importances, yerr=std_importances,
            fmt='b-', label='Feature Importance',
            alpha=0.5, ecolor='lightblue')

# Plot thresholds
plt.axhline(original_threshold, color='r', linestyle='--',
           label=f'Original Threshold ({original_threshold:.4f})')
plt.axhline(relaxed_threshold, color='g', linestyle='--',
           label=f'44-Feature Threshold ({relaxed_threshold:.4f})')

# Fill between areas
plt.fill_between(x, importances, relaxed_threshold,
                where=(importances >= relaxed_threshold),
                color='green', alpha=0.1, label='Selected Features')

# Annotations
plt.text(len(importances)*0.7, original_threshold*1.1,
        f'Original: {len(original_selected)} features', color='red')
plt.text(len(importances)*0.7, relaxed_threshold*0.9,
        f'Selected: {len(relaxed_selected)} features', color='green')

plt.title('Feature Importance Threshold Comparison\n(Targeting 44 Features)')
plt.xlabel('Feature Rank')
plt.ylabel('Mean Importance Score')
plt.legend()
plt.grid(True)
plt.show()

# Print analysis results
print("\n=== Threshold Analysis ===")
print(f"Original threshold: {original_threshold:.4f} ({len(original_selected)} features)")
print(f"44-feature threshold: {relaxed_threshold:.4f} ({len(relaxed_selected)} features)")

# Calculate importance coverage
total_importance = stability_scores['Mean_Importance'].sum()
original_importance = original_selected['Mean_Importance'].sum()
relaxed_importance = relaxed_selected['Mean_Importance'].sum()

print("\n=== Importance Coverage ===")
print(f"Original selection: {(original_importance/total_importance)*100:.2f}%")
print(f"44-feature selection: {(relaxed_importance/total_importance)*100:.2f}%")

# Format and display selected features
print("\n=== Selected Features ===")
selected_features_df = relaxed_selected[['Feature', 'Mean_Importance', 'Std_Importance', 'Relative_Importance']].copy()
selected_features_df['Mean_Importance'] = selected_features_df['Mean_Importance'].apply(lambda x: f"{x:.4f}")
selected_features_df['Std_Importance'] = selected_features_df['Std_Importance'].apply(lambda x: f"{x:.4f}")
selected_features_df['Relative_Importance'] = selected_features_df['Relative_Importance'].apply(lambda x: f"{x:.2f}%")
selected_features_df.columns = ['Feature', 'Mean Imp.', 'Std Imp.', 'Relative Imp.']
print(selected_features_df.to_string(index=False))

# Compare with top 44
top_44 = stability_scores.head(44)
overlap = set(relaxed_selected['Feature']).intersection(set(top_44['Feature']))
print(f"\n=== Comparison with Top 44 ===")
print(f"Features in common with top 44: {len(overlap)}")
print(f"Overlap percentage: {(len(overlap)/44)*100:.2f}%")

# Show differences with top 44
different_features = set(relaxed_selected['Feature']).symmetric_difference(set(top_44['Feature']))
if different_features:
    print("\n=== Features Different from Top 44 ===")

    # Features in threshold selection but not in top 44
    extra_features = set(relaxed_selected['Feature']) - set(top_44['Feature'])
    if extra_features:
        print("\nIn threshold selection but not in top 44:")
        extra_df = stability_scores[stability_scores['Feature'].isin(extra_features)][
            ['Feature', 'Mean_Importance', 'Std_Importance', 'Relative_Importance']]
        extra_df['Mean_Importance'] = extra_df['Mean_Importance'].apply(lambda x: f"{x:.4f}")
        extra_df['Std_Importance'] = extra_df['Std_Importance'].apply(lambda x: f"{x:.4f}")
        extra_df['Relative_Importance'] = extra_df['Relative_Importance'].apply(lambda x: f"{x:.2f}%")
        extra_df.columns = ['Feature', 'Mean Imp.', 'Std Imp.', 'Relative Imp.']
        print(extra_df.to_string(index=False))

    # Features in top 44 but not in threshold selection
    missing_features = set(top_44['Feature']) - set(relaxed_selected['Feature'])
    if missing_features:
        print("\nIn top 44 but not in threshold selection:")
        missing_df = stability_scores[stability_scores['Feature'].isin(missing_features)][
            ['Feature', 'Mean_Importance', 'Std_Importance', 'Relative_Importance']]
        missing_df['Mean_Importance'] = missing_df['Mean_Importance'].apply(lambda x: f"{x:.4f}")
        missing_df['Std_Importance'] = missing_df['Std_Importance'].apply(lambda x: f"{x:.4f}")
        missing_df['Relative_Importance'] = missing_df['Relative_Importance'].apply(lambda x: f"{x:.2f}%")
        missing_df.columns = ['Feature', 'Mean Imp.', 'Std Imp.', 'Relative Imp.']
        print(missing_df.to_string(index=False))

# Save results with all metrics
output_df = relaxed_selected[['Feature', 'Mean_Importance', 'Std_Importance', 'Relative_Importance']]
output_df.to_csv('threshold_44_features_with_metrics.csv', index=False)

In [ ]:
# Add VB classification to the selected features dataset
def add_vb_classification(df):
    """Add VB_class based on binning"""
    # Create a copy of the DataFrame to avoid the warning
    df_copy = df.copy()

    # Define bins and labels
    vb_bins = [0, 0.08, 0.16, 0.24, 0.32, 0.4, np.inf]
    vb_labels = ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']

    # Create VB_class column
    df_copy.loc[:, 'VB_class'] = pd.cut(df_copy['VB'],
                                       bins=vb_bins,
                                       labels=vb_labels,
                                       include_lowest=True).astype(str)
    return df_copy

# Get the selected features based on different criteria
threshold_selected_features = relaxed_selected['Feature'].tolist()
top_44_features = stability_scores.head(44)['Feature'].tolist()
top_16_features = stability_scores.head(16)['Feature'].tolist()  # Added top 16
top_8_features = stability_scores.head(8)['Feature'].tolist()  # Added top 8

# Create output files with feature sets
def create_output_file(features, filename):
    # Create a copy of the combined_df with selected columns
    output_df = combined_df[meta_cols + features + ['VB']].copy()
    output_df = add_vb_classification(output_df)
    output_path = f'/content/drive/MyDrive/ColabNotebooks/{filename}'
    output_df.to_csv(output_path, index=False)
    print(f"✅ Saved {filename} with {len(features)} features and VB class")
    return output_df

# Save all versions (Including Top 8)
threshold_df = create_output_file(
    threshold_selected_features,
    'Threshold_Selected_Features_with_Metadata.csv'
)

top44_df = create_output_file(
    top_44_features,
    'Top44_Features_with_Metadata.csv'
)

top16_df = create_output_file(  # Added top 16 save
    top_16_features,
    'Top16_Features_with_Metadata.csv'
)

top8_df = create_output_file(  # Added top 8 save
    top_8_features,
    'Top8_Features_with_Metadata.csv'
)

# Print validation samples
print("\n🧪 VB and VB_class samples from top 8:")
print(top8_df[['VB', 'VB_class']].head())  # Added top 8 validation print

print("\n🧪 VB and VB_class samples from top 16:")
print(top16_df[['VB', 'VB_class']].head())

print("\n🧪 VB and VB_class samples from top 44:")
print(top44_df[['VB', 'VB_class']].head())

# Print class distribution for top 8
print("\n📊 VB Class Distribution (Top 8):")
print(top8_df['VB_class'].value_counts())  # Added top 8 class distribution

print("\n📊 VB Class Distribution (Top 16):")
print(top16_df['VB_class'].value_counts())

print("\n📊 VB Class Distribution (Top 44):")
print(top44_df['VB_class'].value_counts())

# Save feature lists separately (Including Top 8)
feature_lists = {
    'threshold_features': threshold_selected_features,
    'top44_features': top_44_features,
    'top16_features': top_16_features,
    'top8_features': top_8_features  # Added top 8
}

for name, features in feature_lists.items():
    with open(f'/content/drive/MyDrive/ColabNotebooks/{name}.txt', 'w') as f:
        f.write('\n'.join(features))
    print(f"✅ Saved feature list to {name}.txt")

# Compare the datasets
print("\n=== Dataset Comparison ===")
print(f"Top 8 features: {len(top_8_features)}")
print(f"Top 16 features: {len(top_16_features)}")
print(f"Top 44 features: {len(top_44_features)}")
print(f"Threshold-based features: {len(threshold_selected_features)}")

# Print feature importance details for top 8
print("\n=== Top 8 Features with Importance Scores ===")
top_8_details = stability_scores.head(8)[['Feature', 'Mean_Importance', 'Std_Importance', 'Relative_Importance']]
top_8_details['Mean_Importance'] = top_8_details['Mean_Importance'].apply(lambda x: f"{x:.4f}")
top_8_details['Std_Importance'] = top_8_details['Std_Importance'].apply(lambda x: f"{x:.4f}")
top_8_details['Relative_Importance'] = top_8_details['Relative_Importance'].apply(lambda x: f"{x:.2f}%")
print(top_8_details.to_string(index=False))

# Calculate importance coverage for each set
total_importance = stability_scores['Mean_Importance'].astype(float).sum()
top8_importance = stability_scores.head(8)['Mean_Importance'].astype(float).sum()
top16_importance = stability_scores.head(16)['Mean_Importance'].astype(float).sum()
top44_importance = stability_scores.head(44)['Mean_Importance'].astype(float).sum()

print("\n=== Importance Coverage ===")
print(f"Top 8 features cover: {(top8_importance/total_importance)*100:.2f}% of total importance")
print(f"Top 16 features cover: {(top16_importance/total_importance)*100:.2f}% of total importance")
print(f"Top 44 features cover: {(top44_importance/total_importance)*100:.2f}% of total importance")

In [ ]:
# Separting data based on material 1 and material 2
import pandas as pd

# Read the original file
df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Top8_Features_with_Metadata.csv')

# Split into material 1 and material 2
material1_df = df[df['material'] == 1]
material2_df = df[df['material'] == 2]

# Save to separate files
material1_path = '/content/drive/MyDrive/ColabNotebooks/Top8_Features_Material1.csv'
material2_path = '/content/drive/MyDrive/ColabNotebooks/Top8_Features_Material2.csv'

material1_df.to_csv(material1_path, index=False)
material2_df.to_csv(material2_path, index=False)

print(f"✅ Saved Material 1 data to {material1_path} with {len(material1_df)} rows")
print(f"✅ Saved Material 2 data to {material2_path} with {len(material2_df)} rows")
A
# Print summary statistics
print("\nMaterial 1 Summary:")
print(material1_df['VB_class'].value_counts())

print("\nMaterial 2 Summary:")
print(material2_df['VB_class'].value_counts())

In [ ]:

import pandas as pd

# Read the Material 1 file
material1_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Top8_Features_Material1.csv')

# Create a pivot table showing counts of each VB_class per case
class_counts = material1_df.pivot_table(
    index='case',
    columns='VB_class',
    values='VB',  # Using VB column just for counting (could use any column)
    aggfunc='count',
    fill_value=0
)

# Reorder columns to match the wear progression
column_order = ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']
class_counts = class_counts[column_order]

# Add a total row
class_counts.loc['Total'] = class_counts.sum()

# Display the table with some formatting
print("VB Class Distribution by Case (Material 2)")
print("----------------------------------------")
print(class_counts.to_string())

# Save to CSV if needed
class_counts.to_csv('/content/drive/MyDrive/ColabNotebooks/Material1_ClassCounts_by_Case.csv')
print("\n✅ Saved class counts by case to Material1_ClassCounts_by_Case.csv")

In [ ]:
#finding the ranges for classes per case (for cgan model)
import pandas as pd

# Read the Material 1 file
material1_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Top8_Features_Material1.csv')

# Create the pivot table (same as before)
class_counts = material1_df.pivot_table(
    index='case',
    columns='VB_class',
    values='VB',
    aggfunc='count',
    fill_value=0
)[['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']]

# Calculate ranges for each class
ranges = pd.DataFrame({
    'Class': ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail'],
    'Range': [
        f"[{class_counts['fresh'].min()}, {class_counts['fresh'].max()}]",
        f"[{class_counts['earlyworn'].min()}, {class_counts['earlyworn'].max()}]",
        f"[{class_counts['midworn'].min()}, {class_counts['midworn'].max()}]",
        f"[{class_counts['worn'].min()}, {class_counts['worn'].max()}]",
        f"[{class_counts['highworn'].min()}, {class_counts['highworn'].max()}]",
        f"[{class_counts['fail'].min()}, {class_counts['fail'].max()}]"
    ],
    'Min': class_counts.min().values,
    'Max': class_counts.max().values
})

print("VB Class Count Ranges by Case (Material 1)")
print("-----------------------------------------")
print(ranges.to_string(index=False))


In [ ]:
material1_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Top8_Features_Material1.csv')

# Calculate run ranges per case
run_ranges = material1_df.groupby('case')['run'].agg(['min', 'max', 'count'])
run_ranges['range'] = run_ranges['max'] - run_ranges['min'] + 1  # +1 because both endpoints inclusive
run_ranges['run_sequence'] = material1_df.groupby('case')['run'].apply(list)

# Format the output
run_ranges = run_ranges.rename(columns={
    'min': 'First Run',
    'max': 'Last Run',
    'count': 'Total Runs',
    'range': 'Run Span'
})

print("Run Number Ranges by Case (Material 1)")
print("-------------------------------------")
print(run_ranges[['First Run', 'Last Run', 'Run Span', 'Total Runs']].to_string())

In [ ]:
#Randomly selecting 6 cases for training file and 2 cases for testing file
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Load Material 1 data
material1_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Top8_Features_Material1.csv')

# Get unique case numbers
unique_cases = material1_df['case'].unique()

# Randomly select 6 cases for training and 2 for testing
train_cases = np.random.choice(unique_cases, size=6, replace=False)
test_cases = np.array([case for case in unique_cases if case not in train_cases])

# Split the data
train_df = material1_df[material1_df['case'].isin(train_cases)]
test_df = material1_df[material1_df['case'].isin(test_cases)]

# Save to files
train_path = '/content/drive/MyDrive/ColabNotebooks/Material1_Train_6cases.csv'
test_path = '/content/drive/MyDrive/ColabNotebooks/Material1_Test_2cases.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

# Print summary
print(f"✅ Training file saved with {len(train_df)} rows from cases: {sorted(train_cases)}")
print(f"✅ Testing file saved with {len(test_df)} rows from cases: {sorted(test_cases)}")

# Show class distribution in both sets
print("\nClass Distribution in Training Data:")
print(train_df['VB_class'].value_counts())

print("\nClass Distribution in Testing Data:")
print(test_df['VB_class'].value_counts())

In [ ]:
#CGAN MODEL
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random
import math
from torch.utils.data import DataLoader, Dataset
# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Load and preprocess the data
data = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Material1_Train_6cases.csv')

SENSOR_FEATURES = [
    'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
    'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
    'vib_spindle_mag_centroid', 'AE_table_peak'
]

def normalize_features(df):
    min_doc, max_doc = 0.2, 1.8
    min_feed, max_feed = 0.05, 0.6
    min_time, max_time = 0, int(df['time'].max())
    df_normalized = df.copy()
    df_normalized['DOC'] = (df['DOC'] - min_doc) / (max_doc - min_doc)
    df_normalized['feed'] = (df['feed'] - min_feed) / (max_feed - min_feed)
    df_normalized['time'] = (df['time'] - min_time) / (max_time - min_time)
    for feature in SENSOR_FEATURES:
        min_val = df[feature].min()
        max_val = df[feature].max()
        df_normalized[feature] = (df[feature] - min_val) / (max_val - min_val)
    return df_normalized, {
        'time': (min_time, max_time),
        'sensor_ranges': {feature: (df[feature].min(), df[feature].max()) for feature in SENSOR_FEATURES}
    }

normalized_data, feature_ranges = normalize_features(data)

class ToolWearDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.wear_classes = ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        features = torch.tensor([row['DOC'], row['feed'], row['time']] +
                              [row[feature] for feature in SENSOR_FEATURES], dtype=torch.float)
        wear_class = row['VB_class']
        condition = torch.zeros(6)
        condition[self.wear_classes.index(wear_class)] = 1
        return features, condition

class Generator(nn.Module):
    def __init__(self, latent_dim, condition_dim):
        super(Generator, self).__init__()
        def block(in_feat, out_feat):
            return [nn.Linear(in_feat, out_feat), nn.LeakyReLU(0.2), nn.BatchNorm1d(out_feat, 0.8)]
        self.model = nn.Sequential(
            *block(latent_dim + condition_dim, 512),
            *block(512, 1024),
            *block(1024, 512),
            *block(512, 256),
            nn.Linear(256, 11),
            nn.Sigmoid()
        )

    def forward(self, noise, labels):
        x = torch.cat((noise, labels), -1)
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, feature_dim, condition_dim):
        super(Discriminator, self).__init__()
        def block(in_feat, out_feat):
            return [nn.Linear(in_feat, out_feat), nn.LeakyReLU(0.2), nn.Dropout(0.3)]
        self.model = nn.Sequential(
            *block(feature_dim + condition_dim, 512),
            *block(512, 512),
            *block(512, 256),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, features, labels):
        x = torch.cat((features, labels), -1)
        return self.model(x)

latent_dim = 128
condition_dim = 8  # 6 class + 2 (doc, feed)
feature_dim = 11
batch_size = 32
num_epochs = 10000

generator = Generator(latent_dim, condition_dim)
discriminator = Discriminator(feature_dim, condition_dim)

criterion = nn.BCELoss()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

dataset = ToolWearDataset(normalized_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

def train_cgan():
    for epoch in range(num_epochs):
        for i, (real_features, labels) in enumerate(dataloader):
            batch_size = real_features.size(0)
            valid = torch.ones(batch_size, 1)
            fake = torch.zeros(batch_size, 1)

            g_optimizer.zero_grad()
            z = torch.randn(batch_size, latent_dim)
            doc = torch.rand(batch_size, 1)
            feed = torch.rand(batch_size, 1)
            full_cond = torch.cat((labels, doc, feed), dim=1)
            gen_features = generator(z, full_cond)
            validity = discriminator(gen_features, full_cond)
            g_loss = criterion(validity, valid)
            g_loss.backward()
            g_optimizer.step()

            d_optimizer.zero_grad()
            validity_real = discriminator(real_features, full_cond)
            d_real_loss = criterion(validity_real, valid)
            validity_fake = discriminator(gen_features.detach(), full_cond)
            d_fake_loss = criterion(validity_fake, fake)
            d_loss = (d_real_loss + d_fake_loss) / 2
            d_loss.backward()
            d_optimizer.step()

        if epoch % 100 == 0:
            print(f"Epoch {epoch}/{num_epochs} | D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}")

def generate_synthetic_data(num_combinations=20):
    generator.eval()
    synthetic_data = []
    wear_classes = ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']
    class_ranges = {'fresh': (1, 5), 'earlyworn': (1, 5), 'midworn': (0, 5),
                    'worn': (1, 3), 'highworn': (1, 2), 'fail': (1, 2)}

    for i in range(num_combinations):
        doc = round(np.random.uniform(0.75, 1.5), 2)
        feed = round(np.random.uniform(0.25, 0.5), 2)
        doc_norm = (doc - 0.75) / (1.5 - 0.75)
        feed_norm = (feed - 0.25) / (0.5 - 0.25)
        class_idx = random.randint(0, 5)
        condition = torch.zeros(1, 6)
        condition[0, class_idx] = 1
        full_condition = torch.cat((condition, torch.tensor([[doc_norm, feed_norm]])), dim=1)
        z = torch.randn(1, latent_dim)
        with torch.no_grad():
            generated = generator(z, full_condition).numpy()[0]

        num_runs = random.randint(11,16)
        max_case_time = random.randint(40, 80)
        class_counts = {'fail': random.randint(1, 2)}
        remaining_runs = num_runs - class_counts['fail']
        for cls in wear_classes[:-1]:
            if remaining_runs <= 0:
                class_counts[cls] = 0
                continue
            min_c, max_c = class_ranges[cls]
            count = random.randint(min_c, min(max_c, remaining_runs))
            class_counts[cls] = count
            remaining_runs -= count

        sensor_values = {}
        for idx, feature in enumerate(SENSOR_FEATURES):
            min_val, max_val = feature_ranges['sensor_ranges'][feature]
            sensor_values[feature] = generated[idx + 3] * (max_val - min_val) + min_val

        run_counter, current_time = 1, 0
        time_step = max_case_time // (num_runs - 1) if num_runs > 1 else max_case_time
        ordered_classes = [cls for cls in wear_classes for _ in range(class_counts[cls])]

        for wear_class in ordered_classes:
            row = {
                'case': i + 1,
                'run': run_counter,
                'DOC': doc,
                'feed': feed,
                'time': current_time,
                'VB_class': wear_class
            }
            for f, v in sensor_values.items():
                variation = np.random.uniform(-0.05, 0.05) * v
                row[f] = round(v + variation, 6)
            synthetic_data.append(row)
            run_counter += 1
            current_time += time_step

    return pd.DataFrame(synthetic_data)

print("Training CGAN...")
train_cgan()

print("Generating synthetic data...")
syn_df = generate_synthetic_data(20)
syn_df.to_csv('synthetic_data.csv', index=False)

print("Saved synthetic data to synthetic_data.csv")

In [ ]:
#Combine training data with synthetic data in single file for input to evaluation model
import pandas as pd
import numpy as np


# Load the real and synthetic CSV files
real_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Material1_Train_6cases.csv')
synthetic_df = pd.read_csv('/content/synthetic_data.csv')

# Step 1: Find the max case number in real data
max_case_real = real_df['case'].max()

# Step 2: Shift synthetic case numbers to avoid overlap
synthetic_df['case'] = synthetic_df['case'] + max_case_real

# Step 3: Add missing columns to synthetic_df
missing_cols = set(real_df.columns) - set(synthetic_df.columns)
for col in missing_cols:
    synthetic_df[col] = np.nan

# Step 4: Match column order
synthetic_df = synthetic_df[real_df.columns]

# Step 5: Combine datasets
combined_df = pd.concat([real_df, synthetic_df], ignore_index=True)

# Step 6: Save to specified Google Drive path
save_path = '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data.csv'
combined_df.to_csv(save_path, index=False)

print(f"✅ Combined file saved at: {save_path}")


In [ ]:
#visualisation
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
from scipy.stats import wasserstein_distance, ks_2samp

# Define SENSOR_FEATURES if not already defined
SENSOR_FEATURES = [
    'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
    'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
    'vib_spindle_mag_centroid', 'AE_table_peak'
]

def calculate_improved_mape(y_true, y_pred, epsilon=1e-3):
    """
    Calculate MAPE with better handling of zero/near-zero values
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Handle zero/near-zero values
    mask = np.abs(y_true) > epsilon
    if not np.any(mask):
        return 0.0

    # Calculate MAPE only for non-zero values
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mape

def calculate_distribution_metrics(real_vals, synth_vals):
    """
    Calculate distribution similarity metrics
    """
    # Normalize values for comparison
    real_norm = (real_vals - np.mean(real_vals)) / (np.std(real_vals) + 1e-8)
    synth_norm = (synth_vals - np.mean(synth_vals)) / (np.std(synth_vals) + 1e-8)

    # Wasserstein distance
    w_distance = wasserstein_distance(real_norm, synth_norm)

    # Kolmogorov-Smirnov test
    ks_stat, ks_pval = ks_2samp(real_norm, synth_norm)

    return w_distance, ks_stat, ks_pval

def visualize_comparison(real_df, synthetic_df):
    plt.style.use('default')
    sns.set_palette("husl")
    plt.rcParams['figure.figsize'] = [15, 10]

    # Create figure with better layout
    fig = plt.figure(constrained_layout=True)
    gs = fig.add_gridspec(2, 2)

    # 1. DOC vs Feed Distribution with density contours
    ax1 = fig.add_subplot(gs[0, 0])
    sns.kdeplot(data=real_df, x='DOC', y='feed', color='blue', alpha=0.5, levels=5, ax=ax1)
    sns.kdeplot(data=synthetic_df, x='DOC', y='feed', color='red', alpha=0.5, levels=5, ax=ax1)
    ax1.scatter(real_df['DOC'], real_df['feed'], alpha=0.3, label='Real', c='blue', s=20)
    ax1.scatter(synthetic_df['DOC'], synthetic_df['feed'], alpha=0.3, label='Synthetic', c='red', s=20)
    ax1.set_xlabel('DOC')
    ax1.set_ylabel('Feed')
    ax1.set_title('DOC vs Feed Distribution')
    ax1.legend()

    # 2. Wear Class Distribution with percentages
    ax2 = fig.add_subplot(gs[0, 1])
    wear_order = ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']
    real_counts = real_df['VB_class'].value_counts(normalize=True) * 100
    synth_counts = synthetic_df['VB_class'].value_counts(normalize=True) * 100

    x = np.arange(len(wear_order))
    width = 0.35

    ax2.bar(x - width/2, [real_counts.get(c, 0) for c in wear_order], width,
            label='Real', alpha=0.6, color='blue')
    ax2.bar(x + width/2, [synth_counts.get(c, 0) for c in wear_order], width,
            label='Synthetic', alpha=0.6, color='red')

    # Add percentage labels
    for i, v in enumerate([real_counts.get(c, 0) for c in wear_order]):
        ax2.text(i - width/2, v + 1, f'{v:.1f}%', ha='center', va='bottom')
    for i, v in enumerate([synth_counts.get(c, 0) for c in wear_order]):
        ax2.text(i + width/2, v + 1, f'{v:.1f}%', ha='center', va='bottom')

    ax2.set_xlabel('Wear Class')
    ax2.set_ylabel('Percentage (%)')
    ax2.set_title('Wear Class Distribution')
    ax2.set_xticks(x)
    ax2.set_xticklabels(wear_order, rotation=45)
    ax2.legend()

    # 3. Time Series Plot with confidence intervals
    ax3 = fig.add_subplot(gs[1, 0])

    # Plot real data
    for case in real_df['case'].unique()[:3]:
        case_data = real_df[real_df['case'] == case]
        ax3.plot(case_data['time'], case_data['vib_spindle_rms'],
                'b', alpha=0.3, label='Real' if case==1 else None)

    # Plot synthetic data
    for case in synthetic_df['case'].unique()[:3]:
        case_data = synthetic_df[synthetic_df['case'] == case]
        ax3.plot(case_data['time'], case_data['vib_spindle_rms'],
                'r', alpha=0.3, label='Synthetic' if case==1 else None)

    ax3.set_xlabel('Time')
    ax3.set_ylabel('vib_spindle_rms')
    ax3.set_title('Time Series: vib_spindle_rms')
    ax3.legend()

    # 4. PCA with explained variance
    ax4 = fig.add_subplot(gs[1, 1])
    scaler = StandardScaler()
    real_sensors = scaler.fit_transform(real_df[SENSOR_FEATURES])
    synthetic_sensors = scaler.transform(synthetic_df[SENSOR_FEATURES])

    pca = PCA(n_components=2)
    real_pca = pca.fit_transform(real_sensors)
    synthetic_pca = pca.transform(synthetic_sensors)

    # Plot with density contours
    sns.kdeplot(x=real_pca[:, 0], y=real_pca[:, 1], color='blue', alpha=0.5, levels=5, ax=ax4)
    sns.kdeplot(x=synthetic_pca[:, 0], y=synthetic_pca[:, 1], color='red', alpha=0.5, levels=5, ax=ax4)
    ax4.scatter(real_pca[:, 0], real_pca[:, 1], alpha=0.6, label='Real', c='blue', s=20)
    ax4.scatter(synthetic_pca[:, 0], synthetic_pca[:, 1], alpha=0.6, label='Synthetic', c='red', s=20)

    # Add explained variance ratio
    var_ratio = pca.explained_variance_ratio_
    ax4.set_xlabel(f'PC1 ({var_ratio[0]:.1%} var explained)')
    ax4.set_ylabel(f'PC2 ({var_ratio[1]:.1%} var explained)')
    ax4.set_title('PCA of Sensor Features')
    ax4.legend()

    plt.tight_layout()
    plt.show()

    # 5. Sensor Features Box Plots
    plt.figure(figsize=(15, 8))
    for i, feature in enumerate(SENSOR_FEATURES):
        plt.subplot(2, 4, i+1)
        box_data = [real_df[feature].values, synthetic_df[feature].values]
        bplot = plt.boxplot(box_data, labels=['Real', 'Synthetic'],
                          patch_artist=True)

        # Color boxes
        colors = ['lightblue', 'lightcoral']
        for patch, color in zip(bplot['boxes'], colors):
            patch.set_facecolor(color)

        plt.title(feature)
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # 6. Wear Progression Plot
    plt.figure(figsize=(12, 6))
    wear_map = {'fresh': 0, 'earlyworn': 1, 'midworn': 2, 'worn': 3, 'highworn': 4, 'fail': 5}
    real_df['wear_numeric'] = real_df['VB_class'].map(wear_map)
    synthetic_df['wear_numeric'] = synthetic_df['VB_class'].map(wear_map)

    # Plot wear progression
    for case in real_df['case'].unique()[:3]:
        case_data = real_df[real_df['case'] == case]
        plt.plot(case_data['time'], case_data['wear_numeric'],
                'b', alpha=0.3, label='Real' if case==1 else None)

    for case in synthetic_df['case'].unique()[:3]:
        case_data = synthetic_df[synthetic_df['case'] == case]
        plt.plot(case_data['time'], case_data['wear_numeric'],
                'r', alpha=0.3, label='Synthetic' if case==1 else None)

    plt.yticks(range(6), wear_map.keys())
    plt.xlabel('Time')
    plt.ylabel('Wear Class')
    plt.title('Wear Progression Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 7. Correlation Matrix
    plt.figure(figsize=(15, 6))

    # Real data correlation
    plt.subplot(1, 2, 1)
    real_corr = real_df[SENSOR_FEATURES].corr()
    sns.heatmap(real_corr, annot=True, cmap='coolwarm', fmt='.2f',
                square=True, center=0)
    plt.title('Real Data Correlation Matrix')

    # Synthetic data correlation
    plt.subplot(1, 2, 2)
    synth_corr = synthetic_df[SENSOR_FEATURES].corr()
    sns.heatmap(synth_corr, annot=True, cmap='coolwarm', fmt='.2f',
                square=True, center=0)
    plt.title('Synthetic Data Correlation Matrix')

    plt.tight_layout()
    plt.show()

def print_statistical_comparison(real_df, synthetic_df):
    print("\n📊 Enhanced Statistical Comparison Metrics")

    all_features = SENSOR_FEATURES + ['DOC', 'feed', 'time']
    for feature in all_features:
        real_vals = real_df[feature].dropna().values
        synth_vals = synthetic_df[feature].dropna().values

        min_len = min(len(real_vals), len(synth_vals))
        real_vals = real_vals[:min_len]
        synth_vals = synth_vals[:min_len]

        # Basic statistics
        real_mean = real_vals.mean()
        synth_mean = synth_vals.mean()
        mean_diff = abs(real_mean - synth_mean) / abs(real_mean + 1e-8) * 100

        real_std = real_vals.std()
        synth_std = synth_vals.std()
        std_diff = abs(real_std - synth_std) / abs(real_std + 1e-8) * 100

        # Advanced metrics
        rmse = np.sqrt(mean_squared_error(real_vals, synth_vals))
        mape = calculate_improved_mape(real_vals, synth_vals)
        r2 = r2_score(real_vals, synth_vals)

        # Distribution metrics
        w_distance, ks_stat, ks_pval = calculate_distribution_metrics(real_vals, synth_vals)

        print(f"\n🔹 {feature}")
        print(f"  Mean (Real):       {real_mean:.4f} | Synthetic: {synth_mean:.4f} | Diff: {mean_diff:.2f}%")
        print(f"  Std Dev (Real):    {real_std:.4f} | Synthetic: {synth_std:.4f} | Diff: {std_diff:.2f}%")
        print(f"  RMSE:              {rmse:.4f}")
        print(f"  MAPE:              {mape:.2f}%")
        print(f"  R² Score:          {r2:.4f}")
        print(f"  Wasserstein Dist:  {w_distance:.4f}")
        print(f"  KS-test:           stat={ks_stat:.4f}, p={ks_pval:.4f}")

# Example usage
visualize_comparison(data, syn_df)
print_statistical_comparison(data, syn_df)

In [ ]:
#checking Training accuracy
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.spatial.distance import jensenshannon
import matplotlib.pyplot as plt

# ──────────────────────── Load Data ────────────────────────
real_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/Material1_Train_6cases.csv')
synthetic_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data.csv')

# ──────────────────────── Setup ────────────────────────
VB_CLASSES = ['fresh', 'earlyworn', 'midworn', 'worn', 'highworn', 'fail']
SENSOR_FEATURES = [
    'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
    'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
    'vib_spindle_mag_centroid','AE_table_peak'
]
NUM_FEATURES = ['DOC', 'feed', 'time'] + SENSOR_FEATURES

class_mapping = {label: idx for idx, label in enumerate(VB_CLASSES)}
real_df['VB_class_encoded'] = real_df['VB_class'].map(class_mapping)
synthetic_df['VB_class_encoded'] = synthetic_df['VB_class'].map(class_mapping)

X_real = real_df[NUM_FEATURES]
y_real = real_df['VB_class_encoded']
X_syn = synthetic_df[NUM_FEATURES]
y_syn = synthetic_df['VB_class_encoded']

# ───────────── TSTR: Train on Synthetic, Test on Real ─────────────
clf = RandomForestClassifier(random_state=42)
clf.fit(X_syn, y_syn)
y_pred = clf.predict(X_real)

print("🔍 TSTR Evaluation")
print("Accuracy:", accuracy_score(y_real, y_pred))
print("F1-score (macro):", f1_score(y_real, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_real, y_pred, target_names=VB_CLASSES))

# ───────────── Distributional Similarity ─────────────
print("\n📊 Distributional Metrics:")
for col in NUM_FEATURES:
    r = real_df[col].dropna()
    s = synthetic_df[col].dropna()

    # Histogram-based Jensen-Shannon
    r_hist, _ = np.histogram(r, bins=50, density=True)
    s_hist, _ = np.histogram(s, bins=50, density=True)
    r_hist += 1e-8
    s_hist += 1e-8
    js = jensenshannon(r_hist, s_hist)

    # Wasserstein distance
    wd = wasserstein_distance(r, s)

    # Kolmogorov–Smirnov test
    ks_stat, ks_p = ks_2samp(r, s)

    print(f"{col} | JS: {js:.4f} | WD: {wd:.4f} | KS: {ks_stat:.4f} (p={ks_p:.3f})")

# ───────────── Discriminator Test: Real vs Synthetic ─────────────
print("\n🧪 Real vs Synthetic Discriminator Test")

X_combined = pd.concat([X_real, X_syn], ignore_index=True)
y_combined = np.concatenate([np.zeros(len(X_real)), np.ones(len(X_syn))])
disc_model = LogisticRegression(max_iter=1000).fit(X_combined, y_combined)
y_score = disc_model.predict_proba(X_combined)[:, 1]
auc = roc_auc_score(y_combined, y_score)
acc = disc_model.score(X_combined, y_combined)

print(f"AUC: {auc:.4f} → (Lower is better; 0.5 is ideal)")
print(f"Discriminator Accuracy: {acc:.4f}")

# ───────────── Optional: Plot example feature distributions ─────────────
# for feature in ['DOC', 'feed', 'vib_spindle_rms']:
#     plt.figure()
#     real_df[feature].plot(kind='kde', label='Real', linewidth=2)
#     synthetic_df[feature].plot(kind='kde', label='Synthetic', linewidth=2)
#     plt.title(f'Distribution of {feature}')
#     plt.legend()
#     plt.show()


In [ ]:
#Training all the evaluation models
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Reproducibility settings
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

class GANClassifier(Model):
    def __init__(self, num_classes, input_dim=12):
        super().__init__()
        self.num_classes = num_classes
        self.input_dim = input_dim

        # Generator
        self.generator = tf.keras.Sequential([
            layers.Dense(128, input_shape=(64,)),
            layers.LeakyReLU(0.2),
            layers.BatchNormalization(),
            layers.Dense(256),
            layers.LeakyReLU(0.2),
            layers.BatchNormalization(),
            layers.Dense(input_dim, activation='tanh')
        ])

        # Discriminator/Classifier
        self.discriminator = tf.keras.Sequential([
            layers.Dense(256, input_shape=(input_dim,)),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Dense(128),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Dense(1 + num_classes)
        ])

    def call(self, inputs):
        """Implementation of the forward pass"""
        return self.discriminator(inputs)

    def compile(self, g_optimizer, d_optimizer, loss_fn, **kwargs):
        # Add a dummy loss function to satisfy Keras requirements
        super().compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), **kwargs)
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn
        self.d_accuracy_metric = tf.keras.metrics.BinaryAccuracy()
        self.g_accuracy_metric = tf.keras.metrics.BinaryAccuracy()
        self.class_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

    @tf.function
    def train_step(self, data):
        x, y = data
        batch_size = tf.shape(x)[0]
        noise = tf.random.normal([batch_size, 64])

        # Train discriminator
        with tf.GradientTape() as disc_tape:
            # Real data
            d_real = self.discriminator(x)
            real_validity = d_real[:, 0]
            real_class = d_real[:, 1:]

            # Fake data
            fake_data = self.generator(noise)
            d_fake = self.discriminator(fake_data)
            fake_validity = d_fake[:, 0]

            # Discriminator losses
            d_loss_real = self.loss_fn(tf.ones_like(real_validity), real_validity)
            d_loss_fake = self.loss_fn(tf.zeros_like(fake_validity), fake_validity)
            d_loss_gan = (d_loss_real + d_loss_fake) / 2

            # Classification loss
            class_loss = tf.keras.losses.sparse_categorical_crossentropy(
                y, real_class, from_logits=True
            )

            # Total discriminator loss
            d_loss = d_loss_gan + tf.reduce_mean(class_loss)

        # Update discriminator
        d_grads = disc_tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_weights))

        # Train generator
        with tf.GradientTape() as gen_tape:
            fake_data = self.generator(noise)
            d_fake = self.discriminator(fake_data)
            fake_validity = d_fake[:, 0]
            g_loss = self.loss_fn(tf.ones_like(fake_validity), fake_validity)

        # Update generator
        g_grads = gen_tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_weights))

        # Update metrics
        self.d_accuracy_metric.update_state(tf.ones_like(real_validity), tf.sigmoid(real_validity))
        self.d_accuracy_metric.update_state(tf.zeros_like(fake_validity), tf.sigmoid(fake_validity))
        self.g_accuracy_metric.update_state(tf.ones_like(fake_validity), tf.sigmoid(fake_validity))
        self.class_accuracy_metric.update_state(y, real_class)

        return {
            "d_loss": d_loss_gan,
            "g_loss": g_loss,
            "class_loss": tf.reduce_mean(class_loss),
            "d_acc": self.d_accuracy_metric.result(),
            "g_acc": self.g_accuracy_metric.result(),
            "class_acc": self.class_accuracy_metric.result()
        }

    def predict(self, x):
        """Predict class labels"""
        d_out = self.discriminator(x)
        class_logits = d_out[:, 1:]  # Get only classification outputs
        return tf.argmax(class_logits, axis=1)

class ToolWearClassifier:
    def __init__(self):
        self.features = [
            'DOC', 'time', 'feed', 'material',
            'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
            'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
            'vib_spindle_mag_centroid', 'AE_table_peak'
        ]
        self.le = LabelEncoder()
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        self.models = {}
        self.results = []

    def load_data(self, synth_path, test_path):
        """Load and preprocess data"""
        self.synth_df = pd.read_csv(synth_path)
        self.test_df = pd.read_csv(test_path)

        # Encode classes
        self.synth_df['VB_class_encoded'] = self.le.fit_transform(self.synth_df['VB_class'])
        self.test_df['VB_class_encoded'] = self.le.transform(self.test_df['VB_class'])

        # Prepare features
        self.X_train = self.synth_df[self.features]
        self.y_train = self.synth_df['VB_class_encoded']
        self.X_test = self.test_df[self.features]
        self.y_test = self.test_df['VB_class_encoded']

        # Handle missing values and scale
        self.X_train = pd.DataFrame(
            self.imputer.fit_transform(self.X_train),
            columns=self.features
        )
        self.X_test = pd.DataFrame(
            self.imputer.transform(self.X_test),
            columns=self.features
        )

        self.X_train_sc = self.scaler.fit_transform(self.X_train)
        self.X_test_sc = self.scaler.transform(self.X_test)

    def evaluate_model(self, name, y_pred, y_pred_proba=None):
        """Enhanced evaluation metrics"""
        metrics = {
            'Model': name,
            'Accuracy': accuracy_score(self.y_test, y_pred),
            'Precision (Macro)': precision_score(self.y_test, y_pred, average='macro'),
            'Recall (Macro)': recall_score(self.y_test, y_pred, average='macro'),
            'F1-Score (Macro)': f1_score(self.y_test, y_pred, average='macro')
        }

        if y_pred_proba is not None:
            metrics['ROC-AUC (Macro)'] = roc_auc_score(
                tf.keras.utils.to_categorical(self.y_test),
                y_pred_proba,
                average='macro'
            )

        self.results.append(metrics)
        return metrics

    def train_svm(self):
        """Train SVM with optimized parameters"""
        print("Training SVM...")
        self.models['SVM'] = SVC(
            kernel='rbf',
            C=10,
            gamma='scale',
            probability=True,
            random_state=42
        )
        self.models['SVM'].fit(self.X_train_sc, self.y_train)
        y_pred = self.models['SVM'].predict(self.X_test_sc)
        y_pred_proba = self.models['SVM'].predict_proba(self.X_test_sc)
        return self.evaluate_model('SVM', y_pred, y_pred_proba)

    def train_gpc(self):
        """Train GPC with optimized kernel"""
        print("Training GPC...")
        kernel = C(1.0) * RBF(length_scale=1.0)
        self.models['GPC'] = GaussianProcessClassifier(
            kernel=kernel,
            random_state=42,
            n_jobs=-1
        )
        self.models['GPC'].fit(self.X_train_sc, self.y_train)
        y_pred = self.models['GPC'].predict(self.X_test_sc)
        y_pred_proba = self.models['GPC'].predict_proba(self.X_test_sc)
        return self.evaluate_model('GPC', y_pred, y_pred_proba)

    def train_rf(self):
        """Train Random Forest with optimized parameters"""
        print("Training RF...")
        self.models['RF'] = RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        self.models['RF'].fit(self.X_train_sc, self.y_train)
        y_pred = self.models['RF'].predict(self.X_test_sc)
        y_pred_proba = self.models['RF'].predict_proba(self.X_test_sc)
        return self.evaluate_model('RF', y_pred, y_pred_proba)

    def train_gan(self):
        """Train improved GAN classifier"""
        print("Training GAN...")
        num_classes = len(self.le.classes_)
        gan_clf = GANClassifier(num_classes=num_classes, input_dim=len(self.features))

        callbacks = [
            EarlyStopping(
                monitor='class_acc',
                patience=30,
                restore_best_weights=True,
                mode='max'
            ),
            ReduceLROnPlateau(
                monitor='class_acc',
                factor=0.5,
                patience=10,
                min_lr=1e-6,
                mode='max'
            )
        ]

        gan_clf.compile(
            g_optimizer=tf.keras.optimizers.Adam(1e-4, beta_1=0.5),
            d_optimizer=tf.keras.optimizers.Adam(1e-4, beta_1=0.5),
            loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=True)
        )

        history = gan_clf.fit(
            self.X_train_sc, self.y_train,
            epochs=500,
            batch_size=32,
            callbacks=callbacks,
            validation_split=0.2,
            verbose=1
        )

        self.models['GAN'] = gan_clf
        y_pred = gan_clf.predict(self.X_test_sc)
        d_out = gan_clf.discriminator(self.X_test_sc)
        y_pred_proba = tf.nn.softmax(d_out[:, 1:]).numpy()

        return self.evaluate_model('GAN', y_pred, y_pred_proba)

    def plot_results(self):
        """Enhanced visualization of results"""
        # 1. Model Performance Comparison
        results_df = pd.DataFrame(self.results)
        plt.figure(figsize=(12, 6))
        results_df.set_index('Model')[
            ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)']
        ].plot(kind='bar')
        plt.title('Model Performance Comparison')
        plt.ylabel('Score')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        # 2. Confusion Matrices
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        for ax, (name, model) in zip(axes.flatten(), self.models.items()):
            if name == 'GAN':
                y_pred = model.predict(self.X_test_sc)
            else:
                y_pred = model.predict(self.X_test_sc)
            cm = confusion_matrix(self.y_test, y_pred)
            sns.heatmap(
                cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=self.le.classes_,
                yticklabels=self.le.classes_
            )
            ax.set_title(f'{name} Confusion Matrix')
            ax.set_xlabel('Predicted')
            ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()

        # 3. Feature Importance (for RF)
        if 'RF' in self.models:
            plt.figure(figsize=(10, 6))
            importances = self.models['RF'].feature_importances_
            feature_imp = pd.DataFrame({
                'Feature': self.features,
                'Importance': importances
            }).sort_values('Importance', ascending=True)

            feature_imp.plot(
                kind='barh',
                x='Feature',
                y='Importance',
                figsize=(10, 6)
            )
            plt.title('Random Forest Feature Importance')
            plt.tight_layout()
            plt.show()

def main():
    # Initialize classifier
    classifier = ToolWearClassifier()

    try:
        # Load data
        print("Loading and preprocessing data...")
        classifier.load_data(
            '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data40.csv',
            '/content/drive/MyDrive/ColabNotebooks/Material1_Test_2cases.csv'
        )

        # Train all models
        print("\nTraining models...")
        classifier.train_svm()
        classifier.train_gpc()
        classifier.train_rf()
        classifier.train_gan()

        # Plot results
        print("\nGenerating visualizations...")
        classifier.plot_results()

        # Print final results
        results_df = pd.DataFrame(classifier.results)
        print("\nFinal Model Performance:")
        print(results_df.round(4))

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        raise

if __name__ == "__main__":
    main()

In [ ]:
#CNN LSTM model
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

class OptimizedCNNLSTM(Model):
    def __init__(self, input_dim, num_classes):
        super(OptimizedCNNLSTM, self).__init__()
        self.num_classes = num_classes
        self.input_dim = input_dim

        # Optimized hyperparameters
        self.cnn_filters = [64, 128, 256]  # Gradually increasing filters
        self.cnn_dropout = 0.2
        self.lstm_units = [128, 64]  # Gradually decreasing units
        self.lstm_dropout = 0.3
        self.dense_units = [128, 64]  # Gradually decreasing units
        self.dense_dropout = 0.3
        self.learning_rate = 1e-3

        # CNN layers with residual connections
        self.conv_blocks = []
        for filters in self.cnn_filters:
            self.conv_blocks.append([
                layers.Conv1D(filters, 3, padding='same'),
                layers.BatchNormalization(),
                layers.MaxPooling1D(2),
                layers.Dropout(self.cnn_dropout)
            ])

        # LSTM layers
        self.lstm_layers = []
        for units in self.lstm_units:
            self.lstm_layers.append([
                layers.Bidirectional(layers.LSTM(units, return_sequences=True)),
                layers.Dropout(self.lstm_dropout)
            ])

        # Attention mechanism
        self.attention = layers.Dense(1, activation='tanh')

        # Dense layers
        self.dense_blocks = []
        for units in self.dense_units:
            self.dense_blocks.append([
                layers.Dense(units, activation='relu'),
                layers.BatchNormalization(),
                layers.Dropout(self.dense_dropout)
            ])

        self.output_layer = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        # CNN with residual connections
        x = inputs
        for conv_block in self.conv_blocks:
            residual = x
            for layer in conv_block[:-1]:  # Apply all layers except dropout
                x = layer(x)
            x = tf.nn.relu(x)
            if x.shape == residual.shape:  # Add residual if shapes match
                x += residual
            x = conv_block[-1](x, training=training)  # Apply dropout

        # LSTM with skip connections
        for lstm_block in self.lstm_layers:
            residual = x
            x = lstm_block[0](x)
            if x.shape == residual.shape:
                x += residual
            x = lstm_block[1](x, training=training)

        # Attention mechanism
        attention_weights = tf.nn.softmax(self.attention(x), axis=1)
        context_vector = attention_weights * x
        x = tf.reduce_sum(context_vector, axis=1)

        # Dense layers with skip connections
        for dense_block in self.dense_blocks:
            residual = x
            for layer in dense_block:
                x = layer(x, training=training)
            if x.shape == residual.shape:
                x += residual

        return self.output_layer(x)

    def build_graph(self):
        x = layers.Input(shape=(self.input_dim, 1))
        return Model(inputs=[x], outputs=self.call(x))

def train_and_evaluate_single(train_data_path, test_data_path, combination_name):
    """Train and evaluate CNN-LSTM on a single combination"""

    # Define features
    features = [
        'DOC', 'time', 'feed',
        'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
        'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
        'vib_spindle_mag_centroid', 'AE_table_peak'
    ]

    print(f"\n{'='*60}")
    print(f"TRAINING ON: {combination_name}")
    print(f"{'='*60}")

    # Load data
    print("Loading data...")
    train_df = pd.read_csv(train_data_path)
    test_df = pd.read_csv(test_data_path)

    # Preprocessing
    le = LabelEncoder()
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()

    # Combine classes for LabelEncoder
    all_classes = np.union1d(train_df['VB_class'].unique(), test_df['VB_class'].unique())
    le.fit(all_classes)

    # Transform labels
    y_train = le.transform(train_df['VB_class'])
    y_test = le.transform(test_df['VB_class'])

    # Prepare features
    X_train = train_df[features].values
    X_test = test_df[features].values

    # Handle missing values and scale
    print("Preprocessing data...")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # Reshape data
    X_train_reshaped = X_train_sc.reshape(X_train_sc.shape[0], X_train_sc.shape[1], 1)
    X_test_reshaped = X_test_sc.reshape(X_test_sc.shape[0], X_test_sc.shape[1], 1)

    print(f"Training data shape: {X_train_reshaped.shape}")
    print(f"Test data shape: {X_test_reshaped.shape}")

    # Class weights for imbalanced data
    class_weights = compute_class_weight(
        'balanced', classes=np.unique(y_train), y=y_train)
    class_weight_dict = dict(zip(np.unique(y_train), class_weights))

    # Initialize model
    num_classes = len(le.classes_)
    input_dim = len(features)

    # Clear previous models from memory
    tf.keras.backend.clear_session()

    model = OptimizedCNNLSTM(input_dim, num_classes)

    # Compile with optimized learning rate and gradient clipping
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-3,
        clipnorm=1.0,
        beta_1=0.9,
        beta_2=0.999
    )

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks with optimized parameters
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=30,
            restore_best_weights=True,
            min_delta=0.001
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=30,
            min_lr=1e-6,
            verbose=0
        )
    ]

    # Train with optimized batch size and epochs
    print("Training model...")
    history = model.fit(
        X_train_reshaped, y_train,
        epochs=200,
        batch_size=64,
        validation_split=0.2,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=0  # Reduced verbosity for multiple runs
    )

    # Evaluation
    print("Making predictions...")
    y_pred_proba = model.predict(X_test_reshaped, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # Calculate AUC (for multi-class, use ovr strategy)
    try:
        if len(np.unique(y_test)) > 2:
            auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
        else:
            auc = roc_auc_score(y_test, y_pred_proba[:, 1])
    except Exception as e:
        print(f"AUC calculation failed: {e}")
        auc = 0.0

    print(f"\nResults for {combination_name}:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC: {auc:.4f}")

    return {
        'Combination': combination_name,
        'Model': 'CNN-LSTM',
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC': auc
    }, history

def run_all_combinations_and_compare():
    """Run all combinations and create comparison plots"""

    # Define your 4 training combination files
    combination_files = {
        'Combination 20+6': '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data.csv',
        'Combination 30+6': '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data30.csv',
        'Combination 40+6': '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data40.csv',
        'Combination 60+6': '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data60.csv'
    }

    # Test file (assuming same for all)
    test_data_path = '/content/drive/MyDrive/ColabNotebooks/Material1_Test_2cases.csv'

    # Store results
    all_results = []
    all_histories = {}

    # Run each combination
    for combo_name, train_path in combination_files.items():
        try:
            result, history = train_and_evaluate_single(train_path, test_data_path, combo_name)
            all_results.append(result)
            all_histories[combo_name] = history

            # Clear memory between runs
            tf.keras.backend.clear_session()

        except Exception as e:
            print(f"Error with {combo_name}: {str(e)}")
            continue

    # Convert results to DataFrame
    results_df = pd.DataFrame(all_results)

    # Create comparison plots
    create_comparison_plots(results_df, all_histories)

    return results_df, all_histories

def create_comparison_plots(results_df, histories):
    """Create comprehensive comparison plots"""

    # Melt the dataframe for plotting
    df_melted = results_df.melt(
        id_vars=['Combination', 'Model'],
        value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
        var_name='Metric', value_name='Score'
    )

    # Set up plotting style
    plt.style.use('default')
    sns.set_context("notebook", font_scale=1.1)

    # Color palette
    colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

    # 1. Individual metric plots
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']

    for metric in metrics:
        plt.figure(figsize=(12, 6))

        metric_data = df_melted[df_melted['Metric'] == metric]

        bars = plt.bar(range(len(metric_data)), metric_data['Score'],
                      color=colors[0], alpha=0.8, edgecolor='white', linewidth=2)

        plt.title(f'CNN-LSTM {metric} Performance Across Data Combinations',
                 fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Data Combination', fontsize=12, fontweight='semibold')
        plt.ylabel(f'{metric} Score', fontsize=12, fontweight='semibold')
        plt.ylim(0, 1.1)

        # Add value labels
        for i, bar in enumerate(bars):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.3f}', ha='center', va='bottom',
                    fontsize=11, fontweight='bold')

        plt.xticks(range(len(metric_data)),
                  [combo.replace('Combination ', '') for combo in metric_data['Combination']])
        plt.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()

    # 2. Comprehensive comparison plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        ax = axes[i]
        metric_data = df_melted[df_melted['Metric'] == metric]

        bars = ax.bar(range(len(metric_data)), metric_data['Score'],
                     color=colors[0], alpha=0.8, edgecolor='white', linewidth=1.5)

        ax.set_title(f'{metric}', fontsize=14, fontweight='bold', pad=15)
        ax.set_ylabel('Score', fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3, axis='y')

        # Add value labels
        for j, bar in enumerate(bars):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{height:.3f}', ha='center', va='bottom',
                   fontsize=9, fontweight='semibold')

        ax.set_xticks(range(len(metric_data)))
        ax.set_xticklabels([combo.replace('Combination ', '') for combo in metric_data['Combination']],
                          rotation=45, ha='right')

    # Remove empty subplot
    fig.delaxes(axes[5])

    plt.suptitle('CNN-LSTM Performance Comparison Across All Data Combinations',
                fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

    # 3. Training history comparison
    if histories:
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        for combo_name, history in histories.items():
            combo_short = combo_name.replace('Combination ', '')

            # Training accuracy
            ax1.plot(history.history['accuracy'], label=f'{combo_short} - Train', alpha=0.7)
            ax1.plot(history.history['val_accuracy'], label=f'{combo_short} - Val', linestyle='--', alpha=0.7)

        ax1.set_title('Training & Validation Accuracy', fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True, alpha=0.3)

        for combo_name, history in histories.items():
            combo_short = combo_name.replace('Combination ', '')

            # Training loss
            ax2.plot(history.history['loss'], label=f'{combo_short} - Train', alpha=0.7)
            ax2.plot(history.history['val_loss'], label=f'{combo_short} - Val', linestyle='--', alpha=0.7)

        ax2.set_title('Training & Validation Loss', fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True, alpha=0.3)

        # Remove empty subplots
        fig.delaxes(ax3)
        fig.delaxes(ax4)

        plt.suptitle('CNN-LSTM Training History Comparison', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

    # 4. Summary table
    print("\n" + "="*80)
    print("CNN-LSTM PERFORMANCE SUMMARY")
    print("="*80)
    print(results_df.round(4).to_string(index=False))

    # 5. Best performers
    print("\n" + "="*80)
    print("BEST PERFORMING COMBINATIONS BY METRIC")
    print("="*80)

    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']:
        best_idx = results_df[metric].idxmax()
        best_combo = results_df.loc[best_idx, 'Combination']
        best_score = results_df.loc[best_idx, metric]
        print(f"{metric:12s}: {best_combo} ({best_score:.4f})")

# Run the comparison
if __name__ == "__main__":
    print("Starting CNN-LSTM Multi-Combination Training and Comparison...")

    try:
        results_df, histories = run_all_combinations_and_compare()

        # Optional: Save results to CSV
        results_df.to_csv('cnnlstm_combination_results.csv', index=False)
        print("\nResults saved to 'cnnlstm_combination_results.csv'")

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

In [ ]:
#comparison plots
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Your data
data = {
    'Combination': [
        'Combination 30+6', 'Combination 30+6', 'Combination 30+6', 'Combination 30+6',
        'Combination 40+6', 'Combination 40+6', 'Combination 40+6', 'Combination 40+6',
        'Combination 20+6', 'Combination 20+6', 'Combination 20+6', 'Combination 20+6',
        'Combination 60+6', 'Combination 60+6', 'Combination 60+6', 'Combination 60+6',
        'Combination 0+6',  'Combination 0+6',  'Combination 0+6',  'Combination 0+6'
    ],
    'Model': [
        'SVM', 'GPC', 'RF', 'GAN',
        'SVM', 'GPC', 'RF', 'GAN',
        'SVM', 'GPC', 'RF', 'GAN',
        'SVM', 'GPC', 'RF', 'GAN',
        'SVM', 'GPC', 'RF', 'GAN'
    ],
    'Accuracy': [
        0.5000, 0.5000, 0.4000, 0.5667,
        0.4000, 0.4667, 0.4667, 0.4667,
        0.6667, 0.2333, 0.5667, 0.7333,
        0.4667, 0.3000, 0.4667, 0.4667,
        0.5000, 0.4000, 0.4333, 0.5667
    ],
    'Precision': [
        0.5714, 0.2852, 0.2917, 0.5347,
        0.3361, 0.2453, 0.2861, 0.6180,
        0.6679, 0.1083, 0.4931, 0.7991,
        0.4431, 0.1349, 0.3203, 0.5476,
        0.5097, 0.1435, 0.3387, 0.4324
    ],
    'Recall': [
        0.4679, 0.4111, 0.3333, 0.5222,
        0.3651, 0.3833, 0.4028, 0.4456,
        0.6262, 0.2000, 0.5139, 0.7000,
        0.4357, 0.2667, 0.3944, 0.4357,
        0.4956, 0.3333, 0.3968, 0.5139
    ],
    'F1-Score': [
        0.4699, 0.3236, 0.2978, 0.4729,
        0.3444, 0.2860, 0.3209, 0.4327,
        0.6106, 0.1383, 0.4681, 0.6554,
        0.4143, 0.1718, 0.3416, 0.4435,
        0.4943, 0.1953, 0.3539, 0.4367
    ],
    'AUC': [
        0.8208, 0.7842, 0.8592, 0.8786,
        0.8009, 0.7647, 0.8593, 0.8571,
        0.9247, 0.6627, 0.8717, 0.9656,
        0.8707, 0.6930, 0.8602, 0.8792,
        0.8796, 0.6785, 0.8694, 0.8142
    ]
}

# Create DataFrame
df = pd.DataFrame(data)
df['Combination'] = df['Combination'].str.replace('Combination ', '')

# Convert to long format
df_melted = df.melt(id_vars=['Combination', 'Model'],
                   value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
                   var_name='Metric', value_name='Score')

# Set style
plt.style.use('default')
sns.set_context("notebook", font_scale=1.1)

# Custom color palette
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
custom_palette = dict(zip(['SVM', 'GPC', 'RF', 'GAN'], colors))

# Create individual plots for each metric
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']

for metric in metrics:
    fig, ax = plt.subplots(figsize=(12, 6))

    # Filter data for current metric
    metric_data = df_melted[df_melted['Metric'] == metric]

    # Create bar plot with more spacing
    bars = sns.barplot(
        data=metric_data,
        x='Combination', y='Score', hue='Model',
        ax=ax, palette=custom_palette,
        alpha=0.85, edgecolor='white', linewidth=1.5
    )

    # Customize the plot
    ax.set_title(f'{metric} Performance Comparison',
                fontsize=18, fontweight='bold', pad=25)
    ax.set_xlabel('Data Combination', fontsize=14, fontweight='semibold')
    ax.set_ylabel(f'{metric} Score', fontsize=14, fontweight='semibold')
    ax.set_ylim(0, 1.1)

    # Add subtle grid
    ax.grid(True, alpha=0.2, linestyle='--', axis='y')
    ax.set_axisbelow(True)

    # Customize ticks
    ax.tick_params(axis='both', which='major', labelsize=12)

    # Add value labels on bars (only show if height > 0.05 to avoid clutter)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0.05:
            ax.annotate(f'{height:.3f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 5),
                       textcoords="offset points",
                       ha='center', va='bottom',
                       fontsize=10, fontweight='bold',
                       color='black')

    # Customize legend
    legend = ax.legend(title='Model', loc='upper right',
                      frameon=True, fancybox=True, shadow=True)
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_alpha(0.95)
    legend.get_title().set_fontweight('bold')
    legend.get_title().set_fontsize(12)

    # Clean up the plot
    sns.despine()

    plt.tight_layout()
    plt.show()

# Alternative: Create a single comprehensive plot with better spacing
print("\n" + "="*60)
print("COMPREHENSIVE VIEW - ALL METRICS")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]

    # Filter data for current metric
    metric_data = df_melted[df_melted['Metric'] == metric]

    # Create bar plot
    bars = sns.barplot(
        data=metric_data,
        x='Combination', y='Score', hue='Model',
        ax=ax, palette=custom_palette,
        alpha=0.85, edgecolor='white', linewidth=1.2
    )

    # Customize subplot
    ax.set_title(metric, fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Data Combination', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_ylim(0, 1.05)

    # Add grid
    ax.grid(True, alpha=0.2, linestyle='--', axis='y')
    ax.set_axisbelow(True)

    # Add value labels (selective)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0.05:
            ax.annotate(f'{height:.2f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom',
                       fontsize=9, fontweight='semibold')

    # Legend only on first subplot
    if i == 0:
        legend = ax.legend(title='Model', loc='upper left', bbox_to_anchor=(1, 1))
        legend.get_title().set_fontweight('bold')
    else:
        ax.get_legend().remove()

# Remove the empty 6th subplot
fig.delaxes(axes[5])

# Add main title
fig.suptitle('Model Performance Comparison Across Synthetic+Real Data Combinations',
             fontsize=20, fontweight='bold', y=0.98)

plt.tight_layout()
plt.subplots_adjust(top=0.93, right=0.85)
plt.show()

# Create a clean summary table
print("\n" + "="*60)
print("PERFORMANCE SUMMARY TABLE")
print("="*60)

# Create a pivot table for easy reading
summary_table = df.pivot_table(
    index='Combination',
    columns='Model',
    values=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
    aggfunc='first'
)

print(summary_table.round(3).to_string())

# Find best performing model for each metric
print("\n" + "="*60)
print("BEST PERFORMING MODELS BY METRIC")
print("="*60)

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']:
    best_idx = df[metric].idxmax()
    best_model = df.loc[best_idx, 'Model']
    best_combo = df.loc[best_idx, 'Combination']
    best_score = df.loc[best_idx, metric]
    print(f"{metric:10s}: {best_model} with {best_combo} ({best_score:.3f})")

In [ ]:
#base case
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Reproducibility settings
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

class GANClassifier(Model):
    def __init__(self, num_classes, input_dim=12):
        super().__init__()
        self.num_classes = num_classes
        self.input_dim = input_dim

        # Generator
        self.generator = tf.keras.Sequential([
            layers.Dense(128, input_shape=(64,)),
            layers.LeakyReLU(0.2),
            layers.BatchNormalization(),
            layers.Dense(256),
            layers.LeakyReLU(0.2),
            layers.BatchNormalization(),
            layers.Dense(input_dim, activation='tanh')
        ])

        # Discriminator/Classifier
        self.discriminator = tf.keras.Sequential([
            layers.Dense(256, input_shape=(input_dim,)),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Dense(128),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Dense(1 + num_classes)
        ])

    def call(self, inputs):
        """Implementation of the forward pass"""
        return self.discriminator(inputs)

    def compile(self, g_optimizer, d_optimizer, loss_fn, **kwargs):
        # Add a dummy loss function to satisfy Keras requirements
        super().compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), **kwargs)
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn
        self.d_accuracy_metric = tf.keras.metrics.BinaryAccuracy()
        self.g_accuracy_metric = tf.keras.metrics.BinaryAccuracy()
        self.class_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

    @tf.function
    def train_step(self, data):
        x, y = data
        batch_size = tf.shape(x)[0]
        noise = tf.random.normal([batch_size, 64])

        # Train discriminator
        with tf.GradientTape() as disc_tape:
            # Real data
            d_real = self.discriminator(x)
            real_validity = d_real[:, 0]
            real_class = d_real[:, 1:]

            # Fake data
            fake_data = self.generator(noise)
            d_fake = self.discriminator(fake_data)
            fake_validity = d_fake[:, 0]

            # Discriminator losses
            d_loss_real = self.loss_fn(tf.ones_like(real_validity), real_validity)
            d_loss_fake = self.loss_fn(tf.zeros_like(fake_validity), fake_validity)
            d_loss_gan = (d_loss_real + d_loss_fake) / 2

            # Classification loss
            class_loss = tf.keras.losses.sparse_categorical_crossentropy(
                y, real_class, from_logits=True
            )

            # Total discriminator loss
            d_loss = d_loss_gan + tf.reduce_mean(class_loss)

        # Update discriminator
        d_grads = disc_tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_weights))

        # Train generator
        with tf.GradientTape() as gen_tape:
            fake_data = self.generator(noise)
            d_fake = self.discriminator(fake_data)
            fake_validity = d_fake[:, 0]
            g_loss = self.loss_fn(tf.ones_like(fake_validity), fake_validity)

        # Update generator
        g_grads = gen_tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_weights))

        # Update metrics
        self.d_accuracy_metric.update_state(tf.ones_like(real_validity), tf.sigmoid(real_validity))
        self.d_accuracy_metric.update_state(tf.zeros_like(fake_validity), tf.sigmoid(fake_validity))
        self.g_accuracy_metric.update_state(tf.ones_like(fake_validity), tf.sigmoid(fake_validity))
        self.class_accuracy_metric.update_state(y, real_class)

        return {
            "d_loss": d_loss_gan,
            "g_loss": g_loss,
            "class_loss": tf.reduce_mean(class_loss),
            "d_acc": self.d_accuracy_metric.result(),
            "g_acc": self.g_accuracy_metric.result(),
            "class_acc": self.class_accuracy_metric.result()
        }

    def predict(self, x):
        """Predict class labels"""
        d_out = self.discriminator(x)
        class_logits = d_out[:, 1:]  # Get only classification outputs
        return tf.argmax(class_logits, axis=1)

class ToolWearClassifier:
    def __init__(self):
        self.features = [
            'DOC', 'time', 'feed',
            'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
            'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
            'vib_spindle_mag_centroid', 'AE_table_peak'
        ]
        self.le = LabelEncoder()
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        self.models = {}
        self.results = []

    def load_data(self, synth_path, test_path):
        """Load and preprocess data"""
        self.synth_df = pd.read_csv(synth_path)
        self.test_df = pd.read_csv(test_path)

        # Encode classes
        self.synth_df['VB_class_encoded'] = self.le.fit_transform(self.synth_df['VB_class'])
        self.test_df['VB_class_encoded'] = self.le.transform(self.test_df['VB_class'])

        # Prepare features
        self.X_train = self.synth_df[self.features]
        self.y_train = self.synth_df['VB_class_encoded']
        self.X_test = self.test_df[self.features]
        self.y_test = self.test_df['VB_class_encoded']

        # Handle missing values and scale
        self.X_train = pd.DataFrame(
            self.imputer.fit_transform(self.X_train),
            columns=self.features
        )
        self.X_test = pd.DataFrame(
            self.imputer.transform(self.X_test),
            columns=self.features
        )

        self.X_train_sc = self.scaler.fit_transform(self.X_train)
        self.X_test_sc = self.scaler.transform(self.X_test)

    def evaluate_model(self, name, y_pred, y_pred_proba=None):
        """Enhanced evaluation metrics"""
        metrics = {
            'Model': name,
            'Accuracy': accuracy_score(self.y_test, y_pred),
            'Precision (Macro)': precision_score(self.y_test, y_pred, average='macro'),
            'Recall (Macro)': recall_score(self.y_test, y_pred, average='macro'),
            'F1-Score (Macro)': f1_score(self.y_test, y_pred, average='macro')
        }

        if y_pred_proba is not None:
            metrics['ROC-AUC (Macro)'] = roc_auc_score(
                tf.keras.utils.to_categorical(self.y_test),
                y_pred_proba,
                average='macro'
            )

        self.results.append(metrics)
        return metrics

    def train_svm(self):
        """Train SVM with optimized parameters"""
        print("Training SVM...")
        self.models['SVM'] = SVC(
            kernel='rbf',
            C=10,
            gamma='scale',
            probability=True,
            random_state=42
        )
        self.models['SVM'].fit(self.X_train_sc, self.y_train)
        y_pred = self.models['SVM'].predict(self.X_test_sc)
        y_pred_proba = self.models['SVM'].predict_proba(self.X_test_sc)
        return self.evaluate_model('SVM', y_pred, y_pred_proba)

    def train_gpc(self):
        """Train GPC with optimized kernel"""
        print("Training GPC...")
        kernel = C(1.0) * RBF(length_scale=1.0)
        self.models['GPC'] = GaussianProcessClassifier(
            kernel=kernel,
            random_state=42,
            n_jobs=-1
        )
        self.models['GPC'].fit(self.X_train_sc, self.y_train)
        y_pred = self.models['GPC'].predict(self.X_test_sc)
        y_pred_proba = self.models['GPC'].predict_proba(self.X_test_sc)
        return self.evaluate_model('GPC', y_pred, y_pred_proba)

    def train_rf(self):
        """Train Random Forest with optimized parameters"""
        print("Training RF...")
        self.models['RF'] = RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        self.models['RF'].fit(self.X_train_sc, self.y_train)
        y_pred = self.models['RF'].predict(self.X_test_sc)
        y_pred_proba = self.models['RF'].predict_proba(self.X_test_sc)
        return self.evaluate_model('RF', y_pred, y_pred_proba)

    def train_gan(self):
        """Train improved GAN classifier"""
        print("Training GAN...")
        num_classes = len(self.le.classes_)
        gan_clf = GANClassifier(num_classes=num_classes, input_dim=len(self.features))

        callbacks = [
            EarlyStopping(
                monitor='class_acc',
                patience=30,
                restore_best_weights=True,
                mode='max'
            ),
            ReduceLROnPlateau(
                monitor='class_acc',
                factor=0.5,
                patience=10,
                min_lr=1e-6,
                mode='max'
            )
        ]

        gan_clf.compile(
            g_optimizer=tf.keras.optimizers.Adam(1e-4, beta_1=0.5),
            d_optimizer=tf.keras.optimizers.Adam(1e-4, beta_1=0.5),
            loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=True)
        )

        history = gan_clf.fit(
            self.X_train_sc, self.y_train,
            epochs=500,
            batch_size=32,
            callbacks=callbacks,
            validation_split=0.2,
            verbose=1
        )

        self.models['GAN'] = gan_clf
        y_pred = gan_clf.predict(self.X_test_sc)
        d_out = gan_clf.discriminator(self.X_test_sc)
        y_pred_proba = tf.nn.softmax(d_out[:, 1:]).numpy()

        return self.evaluate_model('GAN', y_pred, y_pred_proba)

    def plot_results(self):
        """Enhanced visualization of results"""
        # 1. Model Performance Comparison
        results_df = pd.DataFrame(self.results)
        plt.figure(figsize=(12, 6))
        results_df.set_index('Model')[
            ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)']
        ].plot(kind='bar')
        plt.title('Model Performance Comparison')
        plt.ylabel('Score')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        # 2. Confusion Matrices
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        for ax, (name, model) in zip(axes.flatten(), self.models.items()):
            if name == 'GAN':
                y_pred = model.predict(self.X_test_sc)
            else:
                y_pred = model.predict(self.X_test_sc)
            cm = confusion_matrix(self.y_test, y_pred)
            sns.heatmap(
                cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=self.le.classes_,
                yticklabels=self.le.classes_
            )
            ax.set_title(f'{name} Confusion Matrix')
            ax.set_xlabel('Predicted')
            ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()

        # 3. Feature Importance (for RF)
        if 'RF' in self.models:
            plt.figure(figsize=(10, 6))
            importances = self.models['RF'].feature_importances_
            feature_imp = pd.DataFrame({
                'Feature': self.features,
                'Importance': importances
            }).sort_values('Importance', ascending=True)

            feature_imp.plot(
                kind='barh',
                x='Feature',
                y='Importance',
                figsize=(10, 6)
            )
            plt.title('Random Forest Feature Importance')
            plt.tight_layout()
            plt.show()

def main():
    # Initialize classifier
    classifier = ToolWearClassifier()

    try:
        # Load data
        print("Loading and preprocessing data...")
        classifier.load_data(
            '/content/drive/MyDrive/ColabNotebooks/Material1_Train_6cases.csv',
            '/content/drive/MyDrive/ColabNotebooks/Material1_Test_2cases.csv'
        )

        # Train all models
        print("\nTraining models...")
        classifier.train_svm()
        classifier.train_gpc()
        classifier.train_rf()
        classifier.train_gan()

        # Plot results
        print("\nGenerating visualizations...")
        classifier.plot_results()

        # Print final results
        results_df = pd.DataFrame(classifier.results)
        print("\nFinal Model Performance:")
        print(results_df.round(4))

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        raise

if __name__ == "__main__":
    main()

In [ ]:
#Optimizing the GAN
import os
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
from tensorflow.keras import layers, Model, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
from datetime import datetime

# Set random seeds for reproducibility
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
np.random.seed(42)
tf.random.set_seed(42)

class GANClassifier(Model):
    """
    A GAN-based classifier that combines generative and discriminative learning.
    Now with configurable hyperparameters for testing.
    """
    def __init__(self, num_classes, input_dim=12, latent_dim=64,
                 g_hidden=[128, 256], d_hidden=[256, 128],
                 dropout_rate=0.3, leaky_relu_alpha=0.2):
        super().__init__()
        self.num_classes = num_classes
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.dropout_rate = dropout_rate
        self.leaky_relu_alpha = leaky_relu_alpha

        # Generator network
        self.generator = self._build_generator(g_hidden)

        # Discriminator/Classifier network
        self.discriminator = self._build_discriminator(d_hidden)

        # Metrics trackers
        self.d_accuracy_metric = tf.keras.metrics.BinaryAccuracy()
        self.g_accuracy_metric = tf.keras.metrics.BinaryAccuracy()
        self.class_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

    def _build_generator(self, hidden_units):
        """Build the generator network"""
        model = tf.keras.Sequential()
        model.add(layers.InputLayer(input_shape=(self.latent_dim,)))

        for units in hidden_units:
            model.add(layers.Dense(units))
            model.add(layers.LeakyReLU(self.leaky_relu_alpha))
            model.add(layers.BatchNormalization())

        model.add(layers.Dense(self.input_dim, activation='tanh'))
        return model

    def _build_discriminator(self, hidden_units):
        """Build the discriminator/classifier network"""
        model = tf.keras.Sequential()
        model.add(layers.InputLayer(input_shape=(self.input_dim,)))

        for units in hidden_units:
            model.add(layers.Dense(units))
            model.add(layers.LeakyReLU(self.leaky_relu_alpha))
            model.add(layers.Dropout(self.dropout_rate))

        # Output layer: 1 for real/fake + num_classes for classification
        model.add(layers.Dense(1 + self.num_classes))
        return model

    def call(self, inputs):
        """Forward pass - returns discriminator outputs"""
        return self.discriminator(inputs)

    def compile(self, g_optimizer, d_optimizer, loss_fn, n_critic=1, **kwargs):
        """Configure the training setup"""
        super().compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), **kwargs)
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn
        self.n_critic = n_critic  # Number of critic iterations per generator iteration

    @tf.function
    def train_step(self, data):
        """Single training step"""
        x, y = data
        batch_size = tf.shape(x)[0]

        # Train discriminator (n_critic times)
        for _ in range(self.n_critic):
            # Generate random noise for generator
            noise = tf.random.normal([batch_size, self.latent_dim])

            with tf.GradientTape() as disc_tape:
                # Real data
                d_real = self.discriminator(x)
                real_validity = d_real[:, 0]  # Real/fake output
                real_class = d_real[:, 1:]    # Class outputs

                # Fake data
                fake_data = self.generator(noise)
                d_fake = self.discriminator(fake_data)
                fake_validity = d_fake[:, 0]

                # Discriminator losses
                d_loss_real = self.loss_fn(tf.ones_like(real_validity), real_validity)
                d_loss_fake = self.loss_fn(tf.zeros_like(fake_validity), fake_validity)
                d_loss_gan = (d_loss_real + d_loss_fake) / 2

                # Classification loss
                class_loss = tf.keras.losses.sparse_categorical_crossentropy(
                    y, real_class, from_logits=True
                )

                # Total discriminator loss
                d_loss = d_loss_gan + tf.reduce_mean(class_loss)

            # Update discriminator
            d_grads = disc_tape.gradient(d_loss, self.discriminator.trainable_weights)
            self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_weights))

        # Train generator
        noise = tf.random.normal([batch_size, self.latent_dim])
        with tf.GradientTape() as gen_tape:
            fake_data = self.generator(noise)
            d_fake = self.discriminator(fake_data)
            fake_validity = d_fake[:, 0]
            g_loss = self.loss_fn(tf.ones_like(fake_validity), fake_validity)

        # Update generator
        g_grads = gen_tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_weights))

        # Update metrics
        self.d_accuracy_metric.update_state(tf.ones_like(real_validity), tf.sigmoid(real_validity))
        self.d_accuracy_metric.update_state(tf.zeros_like(fake_validity), tf.sigmoid(fake_validity))
        self.g_accuracy_metric.update_state(tf.ones_like(fake_validity), tf.sigmoid(fake_validity))
        self.class_accuracy_metric.update_state(y, real_class)

        return {
            "d_loss": d_loss_gan,
            "g_loss": g_loss,
            "class_loss": tf.reduce_mean(class_loss),
            "d_acc": self.d_accuracy_metric.result(),
            "g_acc": self.g_accuracy_metric.result(),
            "class_acc": self.class_accuracy_metric.result()
        }

    def predict(self, x):
        """Predict class labels"""
        d_out = self.discriminator(x)
        class_logits = d_out[:, 1:]  # Get only classification outputs
        return tf.argmax(class_logits, axis=1)

    def predict_proba(self, x):
        """Predict class probabilities"""
        d_out = self.discriminator(x)
        class_logits = d_out[:, 1:]
        return tf.nn.softmax(class_logits).numpy()

class GANTrainer:
    """Helper class for training and evaluating the GAN classifier with hyperparameter testing"""
    def __init__(self, features, target_col='VB_class'):
        self.features = features
        self.target_col = target_col
        self.le = LabelEncoder()
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()

    def load_data(self, train_path, test_path):
        """Load and preprocess data"""
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)

        # Encode classes
        train_df['target_encoded'] = self.le.fit_transform(train_df[self.target_col])
        test_df['target_encoded'] = self.le.transform(test_df[self.target_col])

        # Prepare features
        X_train = train_df[self.features]
        y_train = train_df['target_encoded']
        X_test = test_df[self.features]
        y_test = test_df['target_encoded']

        # Handle missing values and scale
        X_train = pd.DataFrame(
            self.imputer.fit_transform(X_train),
            columns=self.features
        )
        X_test = pd.DataFrame(
            self.imputer.transform(X_test),
            columns=self.features
        )

        self.X_train_sc = self.scaler.fit_transform(X_train)
        self.X_test_sc = self.scaler.transform(X_test)
        self.y_train = y_train.values
        self.y_test = y_test.values

        return self.X_train_sc, self.y_train, self.X_test_sc, self.y_test

    def get_test_configurations(self):
        """Generate different hyperparameter configurations to test"""
        return [
            # Architecture variations
            {
                'latent_dim': 32,
                'g_hidden': [64, 128],
                'd_hidden': [128, 64],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            },
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            },
            {
                'latent_dim': 128,
                'g_hidden': [256, 512],
                'd_hidden': [512, 256],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            },

            # Optimization variations
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-3),
                'd_optimizer': optimizers.Adam(1e-3),
                'batch_size': 64
            },
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            },
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-5),
                'd_optimizer': optimizers.Adam(1e-5),
                'batch_size': 16
            },

            # Regularization variations
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.2,
                'n_critic': 3,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            },
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.4,
                'n_critic': 3,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            },

            # Final candidate (default)
            {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.3,
                'n_critic': 1,
                'g_optimizer': optimizers.Adam(1e-4),
                'd_optimizer': optimizers.Adam(1e-4),
                'batch_size': 32
            }
        ]

    def train_gan(self, X_train, y_train, num_classes, input_dim,
                 epochs=500, batch_size=32, validation_split=0.2, config=None):
        """Train the GAN classifier with optional configuration"""
        if config is None:
            config = {
                'latent_dim': 64,
                'g_hidden': [128, 256],
                'd_hidden': [256, 128],
                'dropout_rate': 0.3,
                'leaky_relu_alpha': 0.2
            }

        gan_clf = GANClassifier(
            num_classes=num_classes,
            input_dim=input_dim,
            latent_dim=config.get('latent_dim', 64),
            g_hidden=config.get('g_hidden', [128, 256]),
            d_hidden=config.get('d_hidden', [256, 128]),
            dropout_rate=config.get('dropout_rate', 0.3),
            leaky_relu_alpha=config.get('leaky_relu_alpha', 0.2)
        )

        callbacks = [
            EarlyStopping(
                monitor='val_class_acc',
                patience=30,
                restore_best_weights=True,
                mode='max'
            ),
            ReduceLROnPlateau(
                monitor='val_class_acc',
                factor=0.5,
                patience=10,
                min_lr=1e-6,
                mode='max'
            )
        ]

        gan_clf.compile(
            g_optimizer=config.get('g_optimizer', optimizers.Adam(1e-4, beta_1=0.5)),
            d_optimizer=config.get('d_optimizer', optimizers.Adam(1e-4, beta_1=0.5)),
            loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=True),
            n_critic=config.get('n_critic', 1),
            run_eagerly=True
        )

        history = gan_clf.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=config.get('batch_size', batch_size),
            callbacks=callbacks,
            validation_split=validation_split,
            verbose=1

        )

        return gan_clf, history

    def test_hyperparameters(self, X_train, y_train, X_test, y_test, epochs=100):
        """Test different hyperparameter configurations"""
        num_classes = len(np.unique(y_train))
        input_dim = X_train.shape[1]
        results = []

        for config in self.get_test_configurations():
            print(f"\nTesting configuration: {config}")

            try:
                gan_clf, history = self.train_gan(
                    X_train, y_train,
                    num_classes=num_classes,
                    input_dim=input_dim,
                    epochs=epochs,
                    config=config
                )

                # Evaluate
                metrics = self.evaluate(gan_clf, X_test, y_test)
                metrics['config'] = config
                metrics['epochs_trained'] = len(history.history['class_acc'])

                results.append(metrics)
                print(f"Results: {metrics}")

            except Exception as e:
                print(f"Failed to train with config {config}: {str(e)}")
                continue

        return results

    def analyze_results(self, results):
        """Analyze and compare the results of different configurations"""
        df = pd.DataFrame(results)

        # Save raw results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_file = f'gan_test_results_{timestamp}.csv'
        df.to_csv(results_file, index=False)
        print(f"\nSaved results to {results_file}")

        # Print analysis
        print("\nHyperparameter Analysis:")

        # Architecture impact
        arch_cols = ['latent_dim', 'g_hidden', 'd_hidden']
        if all(col in df.columns for col in arch_cols):
            print("\nArchitecture Impact:")
            print(df.groupby(arch_cols)[['accuracy', 'f1']].mean())

        # Optimization impact
        opt_cols = ['batch_size', 'n_critic']
        if all(col in df.columns for col in opt_cols):
            print("\nOptimization Impact:")
            print(df.groupby(opt_cols)[['accuracy', 'f1']].mean())

        # Find best configuration
        best_idx = df['accuracy'].idxmax()
        best_config = df.loc[best_idx, 'config']
        print(f"\nBest configuration (accuracy={df.loc[best_idx, 'accuracy']:.4f}):")
        def safe_config(config):
          safe_dict = {}
          for k, v in config.items():
              if isinstance(v, tf.keras.optimizers.Optimizer):
                  lr = v.learning_rate
                  if hasattr(lr, 'numpy'):
                      lr_val = lr.numpy()
                  else:
                      lr_val = lr if isinstance(lr, float) else str(lr)
                  safe_dict[k] = f"{v.__class__.__name__}(lr={lr_val})"
              else:
                  safe_dict[k] = v
          return safe_dict
        print(json.dumps(safe_config(best_config), indent=2))

        return df, best_config

    def evaluate(self, model, X_test, y_test):
        """Evaluate model performance"""
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='macro'),
            'recall': recall_score(y_test, y_pred, average='macro'),
            'f1': f1_score(y_test, y_pred, average='macro'),
            'roc_auc': roc_auc_score(
                tf.keras.utils.to_categorical(y_test),
                y_pred_proba,
                average='macro',
                multi_class='ovo'
            )
        }

        # Plot confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=self.le.classes_,
            yticklabels=self.le.classes_
        )
        plt.title('GAN Classifier Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.show()

        return metrics

def main():
    # Example usage
    features = [
        'DOC', 'time', 'feed',
        'vib_spindle_rms', 'vib_spindle_mean', 'vib_spindle_peak',
        'AE_table_mag_centroid', 'smcAC_shape', 'vib_table_skewness',
        'vib_spindle_mag_centroid', 'AE_table_peak'
    ]

    trainer = GANTrainer(features=features, target_col='VB_class')

    # Load data (replace with your actual paths)
    X_train, y_train, X_test, y_test = trainer.load_data(
        '/content/drive/MyDrive/ColabNotebooks/combined_real_synthetic_data.csv',
        '/content/drive/MyDrive/ColabNotebooks/Material1_Test_2cases.csv'
    )

    # Option 1: Test different hyperparameter configurations
    print("Testing different hyperparameter configurations...")
    results = trainer.test_hyperparameters(
        X_train, y_train, X_test, y_test,
        epochs=100  # Fewer epochs for testing
    )

    # Analyze and get best configuration
    results_df, best_config = trainer.analyze_results(results)

    # Option 2: Train with best configuration (or default if not testing)
    print("\nTraining with best configuration...")
    g_optimizer_config = best_config['g_optimizer'].get_config()
    d_optimizer_config = best_config['d_optimizer'].get_config()
    final_config = best_config.copy() # Create a copy to modify
    final_config['g_optimizer'] = tf.keras.optimizers.Adam.from_config(g_optimizer_config)
    final_config['d_optimizer'] = tf.keras.optimizers.Adam.from_config(d_optimizer_config)
    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]
    gan_clf, history = trainer.train_gan(
        X_train, y_train,
        num_classes=num_classes,
        input_dim=input_dim,
        epochs=300,  # More epochs for final training
        batch_size=32,
        config=final_config  # Use best config from testing
    )

    # Evaluate final model
    metrics = trainer.evaluate(gan_clf, X_test, y_test)
    print("\nFinal GAN Classifier Performance:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    # Plot training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['class_acc'], label='Train Accuracy')
    if 'val_class_acc' in history.history:
        plt.plot(history.history['val_class_acc'], label='Validation Accuracy')
    plt.title('Classification Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['d_loss'], label='Discriminator Loss')
    plt.plot(history.history['g_loss'], label='Generator Loss')
    plt.title('GAN Losses')
    plt.legend()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

def display_configurations_table(results_df):
    """Display a table of all tested configurations with their metrics"""

    # Extract hyperparameters from config dictionary
    config_data = []
    for idx, row in results_df.iterrows():
        config = row['config']
        metrics = {
            'Config ID': idx,
            'Latent Dim': config['latent_dim'],
            'G Hidden': '-'.join(map(str, config['g_hidden'])),
            'D Hidden': '-'.join(map(str, config['d_hidden'])),
            'Dropout': config['dropout_rate'],
            'Batch Size': config['batch_size'],
            'Learning Rate': config['g_optimizer'].learning_rate.numpy() if hasattr(config['g_optimizer'].learning_rate, 'numpy')
                            else config['g_optimizer'].learning_rate,
            'n_critic': config['n_critic'],
            'Accuracy': row['accuracy'],
            'F1 Score': row['f1'],
            'Epochs Trained': row['epochs_trained']
        }
        config_data.append(metrics)

    # Create and display DataFrame
    config_table = pd.DataFrame(config_data)

    # Format for better display
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.float_format', '{:.4f}'.format)

    # Sort by accuracy
    config_table = config_table.sort_values('Accuracy', ascending=False)

    # Display the table
    display(config_table)

    return config_table

# Display the configurations table
config_table = display_configurations_table(results_df)

# Now generate the plots (using the fixed plotting code from previous answer)
plot_hyperparameter_results(results_df)